In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs
from functools import partial
os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import DATADIR, DUNCANS_REGIONS_NAMES, MONTH_NAMES, FIGURES, get_region
from jetutils.data import standardize
from jetutils.geospatial import *
from jetutils.jet_finding import *
from jetutils.plots import *
from jetutils.anyspell import make_daily, mask_from_spells_pl, subset_around_onset
import altair as alt
alt.data_transformers.enable("vegafusion")

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

basepath = Path(f"{DATADIR}/exp8")

In [ ]:
ALL_TIMES = (
    pl.datetime_range(
        start=pl.datetime(1959, 1, 1),
        end=pl.datetime(2023, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = (
    ALL_TIMES
    .filter(pl.col("time").dt.month().is_in([6, 7, 8, 9]))
    .filter(pl.col("time").dt.ordinal_day() > 166)
)
summer = summer_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)
big_summer = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9]))
big_summer_daily = big_summer.filter(big_summer["time"].dt.hour() == 0)

summer_doy = summer_daily.dt.ordinal_day().unique()
n_bootstraps = 100

all_jets_one_df = pl.read_parquet(basepath.joinpath("all_jets_one_df.parquet")).cast({"time": pl.Datetime("ms")})

over_europe = pl.col("lon") > -10
lat_over_europe = (pl.col("lat") * pl.col("s")).filter(over_europe).sum() / pl.col("s").filter(over_europe).sum()
lat_over_europe = all_jets_one_df.group_by("time", "jet ID").agg(lat_over_europe.fill_nan(0).alias("lat_over_europe"))

props_uncat = pl.read_parquet(basepath.joinpath("props_as_df_uncat.parquet")).cast({"time": pl.Datetime("ms")})
props_uncat = props_uncat.join(lat_over_europe, on=["time", "jet ID"])

phat_filter = (pl.col("is_polar") < 0.5) | ((pl.col("is_polar") > 0.5) & (pl.col("int") > 1.3e8))
phat_jets = all_jets_one_df.filter((pl.col("is_polar").mean().over(["time", "jet ID"]) < 0.5) | ((pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5) & (pl.col("int").mode().first().over(["time", "jet ID"]) > 1.3e8)))

phat_jets_catd = phat_jets.with_columns(**{"jet ID": (pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5).cast(pl.UInt32())})
phat_props = props_uncat.filter(phat_filter)
phat_props_catd = average_jet_categories(phat_props, polar_cutoff=0.5)
phat_props_catd = phat_props_catd.join(phat_props_catd.rolling("time", period="2d", group_by="jet").agg(**{f"{col}_var": pl.col(col).var() for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]}), on=["time", "jet"])

phat_props_catd_summer = summer_filter.join(phat_props_catd, on="time")

cross_catd_ofile = basepath.joinpath("cross_catd.parquet")
if cross_catd_ofile.is_file():
    cross_catd = pl.read_parquet(cross_catd_ofile)
else:
    cross_catd = track_jets(phat_jets_catd)
    cross_catd.write_parquet(cross_catd_ofile)

pers = pers_from_cross_catd(cross_catd)
pers.write_parquet(basepath.joinpath("jet_persistence.parquet"))
spells_list = spells_from_cross_catd_simple(cross_catd, season=summer, q_STJ=0.705, q_EDJ=0.836, minlen=datetime.timedelta(days=5), smooth=datetime.timedelta(hours=24), fill_holes=datetime.timedelta(hours=18))
for jet, spells in spells_list.items():
    print(jet, spells["spell"].n_unique())
    
daily_spells_list = {a: make_daily(b, "spell", ["len", "spell_of"]) for a, b in spells_list.items()}

## old jets, new jets

In [ ]:
xys = []
all_jets = all_jets_one_df
# .filter(pl.col("int") > 1.1e8)

xys = np.array(xys)
fig, axes = plt.subplots(
    3, 4, figsize=(11, 9), tight_layout=True, sharex="all", sharey="all"
)
axes = axes.ravel()
pair = ["s", "theta", "is_polar"]
cmap = LinearSegmentedColormap.from_list(
    "pinkredpurple", [COLORS[2], COLORS_EXT[10], COLORS[1]]
)
bins = np.linspace(0, 1, 31)
for month in trange(1, 13):
    ax = axes[month - 1]
    X = extract_features(all_jets, pair, season=month)
    probas = X[pair[2]]
    center_stj = X.filter(pl.col("is_polar") < 0.3).mean()
    center_edj = X.filter(pl.col("is_polar") > 0.7).mean()
    X1D = X["is_polar"]

    im1 = ax.hexbin(X[pair[0]], X[pair[1]], cmap=colormaps.gray_r, gridsize=25)
    im2 = ax.hexbin(
        X[pair[0]], X[pair[1]], C=probas, cmap=colormaps.gray_r, gridsize=25
    )

    plt.draw()

    offsets1 = np.asarray(list(map(tuple, im1.get_offsets())), dtype="f, f")
    offsets2 = np.asarray(list(map(tuple, im2.get_offsets())), dtype="f, f")
    mask12 = np.isin(offsets1, offsets2)
    colors = cmap(im2.get_array())
    colors = rgb_to_hsv(colors[:, :3])
    min_s, max_s = 0.0, 1.0
    min_v, max_v = 0.75, 1.0
    scaling = np.clip(
        im1.get_array()[mask12] / im1.get_array()[mask12].max() * 1.1, 0, 1
    )
    f = lambda x: np.sqrt(x)
    colors[:, 1] = min_s + scaling * (max_s - min_s)
    colors[:, 2] = max_v - scaling * (max_v - min_v)
    colors = hsv_to_rgb(colors)
    im2.set_array(None)
    im2.set_facecolor(colors)
    # im2.set_linewidths(0.2)
    im2.set_linewidths(4 * (1 - f(scaling)) ** 2)
    im2.set_edgecolor(colormaps.greys(scaling))
    im2 = ax.add_collection(im2)

    ax.set_xlabel(pair[0])
    ax.set_ylabel(pair[1])
    if pair[0] in ["lev", "theta"]:
        ax.invert_xaxis()
    elif pair[1] in ["lev", "theta"]:
        ax.invert_yaxis()

    ax.set_title(MONTH_NAMES[month - 1])
    ax.scatter(
        *pl.concat([center_stj, center_edj])[pair[:2]].to_numpy().T,
        facecolor="black",
        edgecolor="white",
        marker="X",
        linewidths=1,
        s=45,
    )
    iax = ax.inset_axes([0.65, 0.2, 0.5, 0.5])
    X1D = np.clip(X1D, 0, 1)
    iax.hist(X1D, bins=bins, alpha=0.5, color="black")
    iax.hist(X1D[probas > 0.5], bins=bins, alpha=0.5, color=COLORS[1])
    iax.hist(X1D[probas < 0.5], bins=bins, alpha=0.5, color=COLORS[2])
    iax.set_xticks([0, 0.5, 1])
    iax.set_yticks([])
    iax.spines[["left", "right", "top"]].set_visible(False)
    plt.draw()
fig.savefig(f"{FIGURES}/gmix_demo")

# curvature?

In [ ]:
dlo = central_diff(pl.col("lon")).radians() * RADIUS * pl.col("lat").radians().cos()
dla = central_diff(pl.col("lat")).radians() * RADIUS
dscubed = (dlo ** 2 + dla ** 2).pow(3 / 2)
ddlo = central_diff(central_diff(pl.col("lon"))).radians() * RADIUS * pl.col("lat").radians().cos()
ddla = central_diff(central_diff(pl.col("lat"))).radians() * RADIUS
kappa = (dlo * ddla - dla * ddlo) / dscubed

alpha = pl.arctan2(pl.col("v"), pl.col("u"))
dalpha = central_diff(alpha)
dalpha = pl.when(dalpha.abs() > np.pi).then((2 * np.pi - dalpha.abs()) * -1 * dalpha.sign()).otherwise(dalpha)
dt = haversine_from_dl(pl.col("lat"), central_diff("lon"), central_diff("lat"))

all_jets_one_df = all_jets_one_df.with_columns(
    kappa=kappa.over(["time", "jet ID"]),
    dalphadt=dalpha / dt
)

In [ ]:
for _, this_one in all_jets_one_df.group_by("time", "jet ID"):
    a, b, kappa = this_one["lon", "lat", "dalphadt"]
    plt.scatter(a, b, c=kappa * RADIUS, cmap=colormaps.berlin, vmin=-2 * np.pi, vmax=2 * np.pi)
    if np.random.randint(0, 2) == 1:
        break

# refined hovmoller

In [ ]:
def refined_hovmoller(df: pl.DataFrame, varname: str, min_n: float, max_n: float, times: pl.Series) -> pl.DataFrame:
    varname_ = f"{varname}_interp"
    df = (
        times
        .cast(df.schema["time"])
        .rename("time")
        .to_frame()
        .join(df.filter(pl.col("n") >= min_n, pl.col("n") <= max_n), on="time")
        .group_by("time", "is_polar", "norm_index", maintain_order=True)
        .agg(pl.col(varname_).mean())
    )
    return df


In [ ]:
from matplotlib.colors import Normalize
varname = "PV330"
varname_ = f"{varname}_interp"
factor = 1e6
df = pl.read_parquet(basepath.joinpath(f"{varname}_phat_catd_relative.parquet"))
fig, axes = plt.subplots(8, 4, figsize=(12, 20), constrained_layout=True)
axes = axes.ravel()
spells = spells_list["EDJ"]
n_spells = spells["spell"].n_unique()
min_, max_ = df[varname_].min(), df[varname_].max()
cmap = WERNLI_FLAIR
levels = WERNLI_FLAIR_LEVELS[:-2]
for ax, i in zip(axes, range(n_spells)):
    times = extend_spells(spells_list["EDJ"].filter(pl.col("spell") == i), time_before=datetime.timedelta(days=4))["time"]
    to_plot = refined_hovmoller(df, varname, 0, 5, times)
    to_plot = to_plot.filter(pl.col("is_polar") == True).with_columns(pl.col(varname_) * factor)
    to_plot = squarify(to_plot, ["norm_index", "time"])
    x = to_plot["norm_index"].unique().to_numpy()
    y = np.arange(len(to_plot["time"].unique()))
    z = to_plot[varname_].to_numpy().reshape(len(y), len(x))
    im = ax.contourf(x, y, z, levels=levels, cmap=cmap, extend="max")
    ax.yaxis.set_inverted(True)
    ax.set_title(times[0])
fig.colorbar(im, ax=axes)

In [ ]:
from matplotlib.colors import Normalize
varname = "PV330"
varname_ = f"{varname}_interp"
factor = 1e6
df = pl.read_parquet(basepath.joinpath(f"{varname}_phat_catd_relative.parquet"))
fig, axes = plt.subplots(8, 4, figsize=(12, 20), constrained_layout=True)
axes = axes.ravel()
spells = spells_list["EDJ"]
n_spells = spells["spell"].n_unique()
min_, max_ = df[varname_].min(), df[varname_].max()
cmap = WERNLI_FLAIR
levels = WERNLI_FLAIR_LEVELS[:-2]
for ax, i in zip(axes, range(n_spells)):
    times = extend_spells(spells_list["EDJ"].filter(pl.col("spell") == i), time_before=datetime.timedelta(days=4))["time"]
    to_plot = refined_hovmoller(df, varname, -5, 0, times)
    to_plot = to_plot.filter(pl.col("is_polar") == True).with_columns(pl.col(varname_) * factor)
    to_plot = squarify(to_plot, ["norm_index", "time"])
    x = to_plot["norm_index"].unique().to_numpy()
    y = np.arange(len(to_plot["time"].unique()))
    z = to_plot[varname_].to_numpy().reshape(len(y), len(x))
    im = ax.contourf(x, y, z, levels=levels, cmap=cmap, extend="max")
    ax.yaxis.set_inverted(True)
    ax.set_title(times[0])
fig.colorbar(im, ax=axes)

In [ ]:
from matplotlib.colors import Normalize
varname = "EMFconv250"
varname_ = f"{varname}_interp"
factor = 1
df = pl.read_parquet(basepath.joinpath(f"{varname}_phat_catd_relative.parquet"))
fig, axes = plt.subplots(8, 4, figsize=(12, 20), constrained_layout=True)
axes = axes.ravel()
spells = spells_list["EDJ"]
n_spells = spells["spell"].n_unique()
min_, max_ = df[varname_].quantile(0.05), df[varname_].quantile(0.95)
cmap = colormaps.balance
levels = MaxNLocator(11).tick_values(min_, max_).tolist()
for ax, i in zip(axes, range(n_spells)):
    times = extend_spells(spells_list["EDJ"].filter(pl.col("spell") == i), time_before=datetime.timedelta(days=4))["time"]
    to_plot = refined_hovmoller(df, varname, -5, 5, times)
    to_plot = to_plot.filter(pl.col("is_polar") == True).with_columns(pl.col(varname_) * factor)
    to_plot = squarify(to_plot, ["norm_index", "time"])
    x = to_plot["norm_index"].unique().to_numpy()
    y = np.arange(len(to_plot["time"].unique()))
    z = to_plot[varname_].to_numpy().reshape(len(y), len(x))
    im = ax.contourf(x, y, z, levels=levels, cmap=cmap, extend="both")
    ax.yaxis.set_inverted(True)
    ax.set_title(times[0])
fig.colorbar(im, ax=axes)

In [ ]:
from matplotlib.colors import Normalize
varname = "theta"
varname_ = f"{varname}_interp"
factor = 1
df = pl.read_parquet(basepath.joinpath(f"{varname}_phat_catd_relative.parquet"))
fig, axes = plt.subplots(8, 4, figsize=(12, 20), constrained_layout=True)
axes = axes.ravel()
spells = spells_list["EDJ"]
n_spells = spells["spell"].n_unique()
min_, max_ = df[varname_].min(), df[varname_].max()
cmap = colormaps.amp
levels = MaxNLocator(11).tick_values(315, 350).tolist()
for ax, i in zip(axes, range(n_spells)):
    times = extend_spells(spells_list["EDJ"].filter(pl.col("spell") == i), time_before=datetime.timedelta(days=4))["time"]
    to_plot = refined_hovmoller(df, varname, -5, 5, times)
    to_plot = to_plot.filter(pl.col("is_polar") == True).with_columns(pl.col(varname_) * factor)
    to_plot = squarify(to_plot, ["norm_index", "time"])
    x = to_plot["norm_index"].unique().to_numpy()
    y = np.arange(len(to_plot["time"].unique()))
    z = to_plot[varname_].to_numpy().reshape(len(y), len(x))
    im = ax.contourf(x, y, z, levels=levels, cmap=cmap, extend="both")
    ax.yaxis.set_inverted(True)
    ax.set_title(times[0])
fig.colorbar(im, ax=axes)

# indiv spells

In [ ]:
summer_doy = summer_daily.dt.ordinal_day().unique()
n_bootstraps = 100

cold = pl.col("n") >= 0
warm = pl.col("n") <= 0
reduced = pl.col("n").abs() <= 1e6
mid = [pl.col("norm_index") >= 0.33, pl.col("norm_index") <= 0.66]
entrance = pl.col("norm_index") <= 0.33
exit_ = pl.col("norm_index") >= 0.66

all_region_filters = {
    "cold": [cold, reduced], 
    "warm": [warm, reduced],
    "cold_entrance": [cold, entrance, reduced],
    "warm_entrance": [warm, entrance, reduced],
    "cold_mid": [cold, *mid, reduced],
    "warm_mid": [warm, *mid, reduced],
    "cold_exit": [cold, exit_, reduced],
    "warm_exit": [warm, exit_, reduced],
    "core": [pl.col("n").abs() <= 1e6],
    "warm_far": [pl.col("n") <= -1.2e6, pl.col("norm_index") <= 0.5]
}
filters_for_variables = {
    "APVO": ["cold", "warm"],
    "CPVO": ["cold", "warm"],
    "t2m": ["cold_mid", "warm_mid", "cold", "warm"],
    "tp": ["warm_entrance", "warm_mid", "cold_mid", "cold_exit", "cold", "warm", "warm_far"],
    "theta": ["cold_mid", "warm_mid", "cold_exit", "warm_exit", "cold_entrance", "warm_entrance", "cold", "warm"],
    "EKE250": ["cold", "warm", "core"],
    "EMFconv250": ["cold", "warm", "core"],
}
time_filters = {
    "before": [pl.col("relative_time") < pl.duration(days=0), pl.col("relative_time") >= pl.duration(days=-3)], # only extending to -4 days anyways (next cell)
    "during": [pl.col("relative_time") >= pl.duration(days=0), pl.col("relative_time") <= pl.duration(days=3)],
}

props_to_take = ["mean_lon", "mean_lat", "mean_theta", "mean_s", "tilt", "waviness1", "waviness2", "wavinessR16", "int", "width", "mjo_pc1", "mjo_pc2", "mjo_int", "lat_over_europe"]
do_anom = {"APVO": False, "CPVO": False, "t2m": True, "tp": False, "theta": True, "EKE250": True, "EMFconv250": True} | {prop: True for prop in props_to_take}

relative_dfs = {title: pl.read_parquet(basepath.joinpath(f"{title}meters_relative.parquet")).cast({"time": pl.Datetime("ms")}) for title in filters_for_variables}

clims = {title: df.group_by(pl.col("time").dt.ordinal_day().alias("dayofyear"), "norm_index", "n", "is_polar").agg(pl.col(f"{title}_interp").mean()).sort("is_polar", "dayofyear", "norm_index", "n") for title, df in relative_dfs.items()}
clims_std = {title: df.group_by(pl.col("time").dt.ordinal_day().alias("dayofyear"), "norm_index", "n", "is_polar").agg(pl.col(f"{title}_interp").std()).sort("is_polar", "dayofyear", "norm_index", "n") for title, df in relative_dfs.items()}
clims_sm = {title: clim.with_columns(**{f"{title}_interp": pl.col(f"{title}_interp").filter(pl.col("dayofyear").is_in(summer_doy.implode())).mean().over("is_polar", "n", "norm_index")}) for title, clim in clims.items()}
clims_sstd = {title: clim.with_columns(**{f"{title}_interp": pl.col(f"{title}_interp").filter(pl.col("dayofyear").is_in(summer_doy.implode())).mean().over("is_polar", "n", "norm_index")}) for title, clim in clims_std.items()}

selector = cs.all().exclude("time", "jet")
filter_ = pl.col("time").dt.month().is_in([6, 7, 8, 9])
props_anoms = phat_props_catd.with_columns(((selector - selector.filter(filter_).mean()) / selector.filter(filter_).std()).over("jet"))

mjo = pl.from_pandas(pd.read_fwf(f"{DATADIR}/era5_mjo.txt", header=None, names=["year", "month", "day", "mjo_pc1", "mjo_pc2", "mjo_phase", "mjo_int"], infer_nrows=100000)).with_columns(time=pl.datetime(year=pl.col("year"), month=pl.col("month"), day=pl.col("day"), time_unit="ms")).drop(["year", "month", "day"])
mjo = mjo.with_columns((pl.col("mjo_int") - pl.col("mjo_int").min()) / (pl.col("mjo_int").max() - pl.col("mjo_int").min()) * 100)

grams_wr = pl.read_parquet(f"{DATADIR}/grams_wr.parquet")
grams_wr = grams_wr.cast({"time": pl.Datetime("ms")})
winner_names = pl.DataFrame({"winner": list(range(5)), "name": ["No", "GB", "AL", "AR", "SB"]})
grams_wr = grams_wr.join(winner_names, on="winner")

props_anoms = props_anoms.join(mjo.cast({"time": pl.Datetime("ms")}), on="time")

In [ ]:
everything = {"STJ": None, "EDJ": None}
factors = {"tp": 1000, "APVO": 100, "CPVO": 100}
for jet in ["STJ", "EDJ"]:
    id_ = int(jet == "EDJ")
    dayofyears = daily_spells_list[f"{jet}"]["time"].dt.ordinal_day()
    spells = daily_spells_list[f"{jet}"]
    to_ret = spells["spell"].unique().to_frame()
    to_ret = pl.concat([to_ret.cast({"spell": pl.Int32()}), pl.DataFrame({"spell": [-2, -1]}).cast({"spell": pl.Int32()})])
    everything[jet] = to_ret.clone()
    spells_ext = extend_spells(spells, time_before=datetime.timedelta(days=3))
    for title, these_filters in tqdm(filters_for_variables.items()):
        clim = clims[title]
        clim_sm = clims_sm[title].filter(pl.col("is_polar") == bool(id_))
        clim_std = clims_std[title]
        this_do_anom = do_anom[title]
        space_filters = {
            filter_name: all_region_filters[filter_name] 
            for filter_name in these_filters
        }
        these_filters = {
            f"{filter_name}_{when_}": [*filter_, *time_filter_] 
            for filter_name, filter_ in space_filters.items() 
            for when_, time_filter_ in time_filters.items()
        }
        # ts_bootstrapped = create_bootstrapped_times(spells_ext, summer_daily, n_bootstraps)
        masked = spells_ext.join(relative_dfs[title].with_columns(**{"jet ID": pl.col("is_polar").cast(pl.UInt8())}), on="time")
        varname_ = f"{title}_interp"
        agg = pl.col(varname_).replace([float("inf"), float("-inf"), float("nan")], None).mean().cast(pl.Float32())
        factor = factors.get(title, 1)
        filter_name = list(these_filters)[0]
        if this_do_anom:
            masked = (
                masked
                .with_columns(dayofyear=pl.col("time").dt.ordinal_day())
                .join(clim, on=["is_polar", "dayofyear", "norm_index", "n"])
                .with_columns(pl.col(varname_) - pl.col(f"{varname_}_right"))
                .drop(f"{varname_}_right")
                .join(clim_std, on=["is_polar", "dayofyear", "norm_index", "n"])
                .with_columns(pl.col(varname_) / pl.col(f"{varname_}_right"))
                .drop(f"{varname_}_right", "dayofyear")
            )
            agg_clim = pl.lit(0., dtype=pl.Float32())
        else:
            agg_clim = agg
        for filter_name, filter_ in these_filters.items():
            # results = squarify(results, ["spell"])
            full_name = f"{title}_{filter_name}"
            results = masked.filter(pl.col("jet ID") == id_, *filter_).group_by("spell").agg(**{full_name: agg * factor}).sort("spell").cast({"spell": pl.Int32()})
            # full_name_pvals = f"{full_name}_pvals"
            # results = results.group_by("spell", maintain_order=True).agg(
            #     **{
            #         full_name: pl.col(varname_).last() * factor, 
            #         full_name_pvals: (pl.col(varname_).rank().last() - 1) / n_bootstraps
            #     }
            # ).cast({"spell": pl.Int32()})
            space_filter = space_filters[filter_name.split("_")[0]]
            mean_over_spells = results.select(
                **{
                    "spell": -2,
                    full_name: pl.col(full_name).mean(),
                    # full_name_pvals: pl.col(full_name_pvals).mean(),
                }
            )
            clim_ = clim_sm.filter(pl.col("dayofyear") == 1, *space_filter).select(
                **{
                    "spell": -1,
                    full_name: agg_clim * factor,
                    # full_name_pvals: pl.lit(0.5, dtype=pl.Float32()),
                }
            )
            results = pl.concat([results, mean_over_spells, clim_])
            everything[jet] = everything[jet].join(results, on="spell", how="left")
    for when_, time_filter_ in time_filters.items():
        # masked_props = (
        #     ts_bootstrapped
        #     .join(props_anoms, on="time")
        #     .filter(*time_filter_, pl.col("jet") == jet)
        #     .group_by("sample_index", "spell")
        #     .agg(**{f"{prop}_{when_}": pl.col(prop).mean().cast(pl.Float32()) for prop in props_to_take})
        #     .sort("sample_index", "spell")
        # )
        masked_props = (
            spells_ext
            .join(props_anoms, on="time")
            .filter(*time_filter_, pl.col("jet") == jet)
            .group_by("spell")
            .agg(**{f"{prop}_{when_}": pl.col(prop).mean().cast(pl.Float32()) for prop in props_to_take})
            .sort("spell")
        )
        # theseprops = masked_props.filter(pl.col("sample_index") == n_bootstraps).drop("sample_index")
        mean_over_spells = masked_props.select(cs.all().mean()).with_columns(spell=-2)
        clim_ = pl.DataFrame({"spell": -1, **{f"{name}_{when_}": np.float32(0.) for name in props_to_take}}).cast({"spell": pl.Int32()})
        
        theseprops = pl.concat([masked_props.cast({"spell": pl.Int32()}), mean_over_spells, clim_])
        
        # thesepvals = masked_props.group_by("spell", maintain_order=True).agg(**{f"{prop}_{when_}_pvals": ((pl.col(f"{prop}_{when_}").rank().last() - 1) / n_bootstraps).cast(pl.Float64()) for prop in props_to_take})
        # mean_over_spells = thesepvals.select(cs.all().mean()).with_columns(spell=-2)
        # clim_ = pl.DataFrame({"spell": -1, **{f"{name}_{when_}_pvals": 0.5 for name in props_to_take}}).cast({"spell": pl.Int32()})
        # thesepvals = pl.concat([thesepvals.cast({"spell": pl.Int32()}), mean_over_spells, clim_])
        
        everything[jet] = everything[jet].join(theseprops, on="spell", how="left")
        # everything[jet] = everything[jet].join(thesepvals, on="spell", how="left")
    
    phase_stuff = spells_ext.join(mjo, on="time").group_by("spell").agg(**{f"mjo_phase_{when_}": pl.col("mjo_phase").filter(time_filter_).mode().first() for when_, time_filter_ in time_filters.items()}).sort("spell")
    everything[jet] = everything[jet].join(phase_stuff, on="spell", how="left")
    
    regime_stuff = spells_ext.join(grams_wr, on="time").group_by("spell").agg(
        **{
            f"regime_{when}": pl.col("winner").filter(*time_filter_, pl.col("winner") != 0).mode().first()
            for when, time_filter_ in time_filters.items()
        }
    )
    everything[jet] = everything[jet].join(regime_stuff, on="spell", how="left")
    regime_stuff = spells_ext.join(grams_wr, on="time").group_by("spell").agg(
        **{
            f"regime_withno_{when}": pl.col("winner").filter(time_filter_).mode().first()
            for when, time_filter_ in time_filters.items()
        }
    )
    everything[jet] = everything[jet].join(regime_stuff, on="spell", how="left")
            
everything_stj = everything["STJ"]
everything_edj = everything["EDJ"]

everything_stj.write_parquet(f"{FIGURES}/jet_persistence/figure_data/everything_stj.parquet")
everything_edj.write_parquet(f"{FIGURES}/jet_persistence/figure_data/everything_edj.parquet")

In [ ]:
from pathlib import Path
from string import ascii_uppercase
from sklearn.decomposition import PCA

%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches': "tight"}
# %config InlineBackend.print_figure_kwargs = {'bbox_inches': None}


everything_stj = pl.read_parquet(f"{FIGURES}/jet_persistence/figure_data/everything_stj.parquet")
everything_edj = pl.read_parquet(f"{FIGURES}/jet_persistence/figure_data/everything_edj.parquet")
winner_names = pl.DataFrame({"winner": list(range(5)), "name": ["No", "GB", "AL", "AR", "SB"]})

plt.rc("xtick", labelsize=12)
plt.rc("ytick", labelsize=12)
plt.rc("savefig", bbox="tight")
# plt.rc("savefig", bbox=None)
when = "during"

relevant_keys_STJ = {
    "mean_lat_before": ["anom", PRETTIER_VARNAME["mean_lat"] + ", before", r"$\sigma$"],
    # "theta2PVU_cold_before": ["anom", r"$\theta$ at 2PVU, cold, before", r"$\sigma$"],
    # "theta2PVU_warm_before": ["anom", r"$\theta$ at 2PVU, warm, before", r"$\sigma$"],
    "t2m_cold_before": ["anom", "2m temp., cold, before", r"$\sigma$"],
    # "t2m_warm_before": ["anom", "2m temp., warm, before", r"$\sigma$"],
    "mean_lat_during": ["anom", PRETTIER_VARNAME["mean_lat"], r"$\sigma$"],
    # "theta2PVU_cold_during": ["anom", r"$\theta$ at 2PVU, cold", r"$\sigma$"],
    # "theta2PVU_warm_during": ["anom", r"$\theta$ at 2PVU, warm", r"$\sigma$"],
    "t2m_cold_during": ["anom", "2m temp., cold", r"$\sigma$"],
    # "EKE250_cold_during": ["anom", "EKE, cold", r"$\sigma$"],
    # "t2m_warm_during": ["anom", "2m temp., warm", r"$\sigma$"],
    # "tp_cold_during": ["abs", "Prec., cold", r"mm"], 
    "tp_warm_far_before": ["abs", "Prec., warm, far, before", r"mm"], 
    "tp_warm_far_during": ["abs", "Prec., warm, far", r"mm"], 
    "APVO_cold_during": ["zeroone", "AWB, cold", r"$\%$"], 
    "APVO_warm_during": ["zeroone", "AWB, warm", r"$\%$"], 
    "CPVO_cold_during": ["zeroone", "CWB, cold", r"$\%$"], 
}

pvals_STJ = [f"{key}_pvals" for key in relevant_keys_STJ]
relevant_keys_STJ = {
    key: [type_, f"{val} [{unit}]"] for key, (type_, val, unit) in relevant_keys_STJ.items()
}

relevant_keys_EDJ = {
    "mean_lat_before": ["anom", PRETTIER_VARNAME["mean_lat"] + ", before", r"$\sigma$"],
    # "theta2PVU_cold_before": ["anom", r"$\theta$ at 2PVU, cold, before", r"$\sigma$"],
    # "theta2PVU_warm_before": ["anom", r"$\theta$ at 2PVU, warm, before", r"$\sigma$"],
    "mean_lat_during": ["anom", PRETTIER_VARNAME["mean_lat"], r"$\sigma$"],
    "lat_over_europe_during": ["anom", "Latitude over Europe", r"$\sigma$"],
    "wavinessR16_during": ["anom", PRETTIER_VARNAME["wavinessR16"], r"$\sigma$"],
    # "theta2PVU_cold_during": ["anom", r"$\theta$ at 2PVU, cold", r"$\sigma$"],
    # "theta2PVU_warm_during": ["anom", r"$\theta$ at 2PVU, warm", r"$\sigma$"],
    # "t2m_cold_during": ["anom", "2m temp., cold", r"$\sigma$"],
    # "t2m_warm_during": ["anom", "2m temp., warm", r"$\sigma$"],
    # "EMFconv250_core_during": ["anom", "EMF conv. at jet core", r"$\sigma$"],
    "EKE250_core_during": ["anom", "EKE, core", r"$\sigma$"],
    # "EKE250_warm_during": ["anom", "EKE, warm", r"$\sigma$"],
    "tp_warm_entrance_before": ["abs", "Prec., warm entr., before", r"mm"], 
    "tp_warm_entrance_during": ["abs", "Prec., warm entr.", r"mm"],
    # "zblocks_warm_before": ["zeroone", "Blocks, warm, before", r"$\%$"], 
    "CPVO_cold_before": ["zeroone", "CWB, cold, before", r"$\%$"], 
    # "SCPVS_warm_before": ["zeroone", "CWB, warm, before", r"$\%$"], 
    # "zblocks_warm_during": ["zeroone", "Blocks, warm flank", r"$\%$"], 
    "CPVO_cold_during": ["zeroone", "CWB, cold", r"$\%$"], 
    "APVO_warm_during": ["zeroone", "AWB, warm", r"$\%$"], 
    # "SCPVS_warm_during": ["zeroone", "CWB, warm flank", r"$\%$"], 
    # "SAPVS_warm_during": ["zeroone", "AWB, warm flank", r"$\%$"], 
}

pvals_EDJ = [f"{key}_pvals" for key in relevant_keys_EDJ]
relevant_keys_EDJ = {
    key: [type_, f"{val} [{unit}]"] for key, (type_, val, unit) in relevant_keys_EDJ.items()
}

# danom = 0.6
levels_anom = np.array([-1.4, -1.0, -0.6, -0.2, 0.2, 0.6, 1.0, 1.4])
levels_anom = levels_anom[np.round(levels_anom, 10) != 0]
plot_kwargs = {
    "anom": {
        "cmap": colormaps.BlWhRe, 
        "norm": BoundaryNorm(levels_anom, colormaps.BlWhRe.N, extend="neither"),
        "cbar_ticks": levels_anom,
    },
    "abs": {
        "cmap": colormaps.lapaz_r, 
        "norm": BoundaryNorm([0, 1, 2, 3, 4], colormaps.lapaz_r.N, extend="max"),
        "cbar_ticks": [0, 2, 4],
    },
    "zeroone": {
        "cmap": colormaps.torch_r, 
        "norm": BoundaryNorm([0, 10, 20, 30, 40, 50, 60, 70], colormaps.torch_r.N, extend="max"),
        "cbar_ticks": [0, 20, 40, 60],
    },
}
order_i_want = ["zeroone", "abs", "anom"]
def signif_array(X_pvals: pl.DataFrame, type_: str, alpha: float = 0.05):
    ouais = X_pvals.to_numpy().T[ys[i][:-1]]
    if type_ == "anom":
        ouais = (ouais > 1 - alpha / 2) | (ouais < alpha / 2)
    else:
        ouais = ouais > 1 - alpha
    dummy = np.ones_like(ouais, dtype=float)
    return np.ma.masked_array(dummy, mask=~ouais)

early_summer = ALL_TIMES.filter(pl.col("time").dt.month() == 6, pl.col("time").dt.day() >= 15, pl.col("time").dt.day() <= 25)["time"]
late_summer = ALL_TIMES.filter(pl.col("time").dt.month() == 9, pl.col("time").dt.day() >= 20)["time"]

spells_in_early_or_late = {
    jet: spells_list[f"{jet}"]
    .group_by("spell", maintain_order=True)
    .agg(
        is_early=(pl.col("time").is_in(early_summer.implode()).mean() > 0.5) 
        | (pl.col("time").is_in(late_summer.implode()).mean() > 0.5) 
    )
    for jet in ["STJ", "EDJ"]
}

estj = everything_stj.join(spells_in_early_or_late["STJ"], on="spell", how="left")
filters_stj = {
    "early_late": ~pl.col("is_early"),
    # "warm_cold": (pl.col("t2m_cold_during") > 0.2),
    "enhanced_precip": pl.col("tp_warm_far_during") > 2,
    # "enhanced_ccwb": pl.col("CPVO_cold_during") > 1.1 * pl.col("CPVO_cold_during").last(),
    # "enhanced_cawb": pl.col("APVO_cold_during") > 1.1 * pl.col("APVO_cold_during").last(),
    "t2m_cold_during": "t2m_cold_during",
}
filters_stj = estj.select(pl.col("spell"), **filters_stj)[:-2]

groups_stj = filters_stj.select(pl.col("spell"), "t2m_cold_during", group=pl.concat_str(cs.boolean().cast(pl.UInt8())).str.to_integer(base=2))
order_stj = groups_stj.select(order=pl.col("spell").sort_by("group", "t2m_cold_during"))["order"].to_numpy()

# group_1 = filters_stj["early_late"].arg_true()
# group_2 = filters_stj.select(hoho=pl.col("enhanced_precip") & ~pl.col("early_late") & ~pl.col("enhanced_cawb") & ~pl.col("enhanced_ccwb"))["hoho"].arg_true()
# group_2_and_3 = filters_stj.select(hoho=(pl.col("enhanced_ccwb") | pl.col("enhanced_cawb")) & ~pl.col("early_late") & pl.col("enhanced_precip"))["hoho"].arg_true()
# group_3 = filters_stj.select(hoho=(pl.col("enhanced_ccwb") | pl.col("enhanced_cawb")) & ~pl.col("early_late") & ~pl.col("enhanced_precip"))["hoho"].arg_true()
# group_last = filters_stj.select(hoho=pl.all_horizontal(cs.boolean().not_()))["hoho"].arg_true()
# order_stj = np.concatenate([group_1, group_2, group_2_and_3, group_3, group_last])

filters_edj = {
    # "south": (pl.col("mean_lat_during") < -0.1),
    "enhanced_cwb": (pl.col("CPVO_cold_during") > 40) | (pl.col("CPVO_cold_before") > 40),
    "enhanced_precip": pl.col("tp_warm_entrance_during") > 4,
    "lat_over_europe_during": pl.col("lat_over_europe_during"),
    # "europe_highlat": pl.col("lat_over_europe_during") > 0,
    # "enhanced_ccwb": pl.col("CPVO_cold_during") > pl.col("CPVO_cold_during").last(),
}
filters_edj = everything_edj.select(pl.col("spell"), **filters_edj)[:-2]
groups_edj = filters_edj.select(pl.col("spell"), "lat_over_europe_during", group=pl.concat_str(cs.boolean().cast(pl.UInt8())).str.to_integer(base=2))
order_edj = groups_edj.select(order=pl.col("spell").sort_by("group", "lat_over_europe_during"))["order"].to_numpy()

# bigX = everything_stj[["APVO_cold_during", "CPVO_cold_during", "t2m_cold_during", "tp_warm_far_during", "mean_lat_during"]][:-2].fill_null(0.)
# bigX = bigX.with_columns((cs.all() - cs.all().mean()) / cs.all().std())
# order_stj = PCA(2).fit_transform(bigX)[:, 0].argsort()

# bigX = everything_edj[["APVO_warm_during", "CPVO_cold_during", "tp_warm_entrance_during", "mean_lat_during"]][:-2].fill_null(0.)
# bigX = bigX.with_columns((cs.all() - cs.all().mean()) / cs.all().std())
# order_edj = PCA(2).fit_transform(bigX)[:, 0].argsort()

size = (11.5, 4.2)
colors_groups = colormaps.pastel([2, 3, 4, 6, 7, 8, 0, 5, 1])
for df, relevant_keys, pvals, jet, groups, order_x in zip([everything_stj, everything_edj], [relevant_keys_STJ, relevant_keys_EDJ], [pvals_STJ, pvals_EDJ], ["STJ", "EDJ"], [groups_stj, groups_edj], [order_stj, order_edj]):
    unique, count = np.unique(np.asarray([ouais[0] for ouais in relevant_keys.values()]), return_counts=True)
    n_cbars = len(unique)
    height_ratios = np.zeros(n_cbars)
    unique_in_order = []
    i = 0
    for ouais in order_i_want:
        if ouais not in unique:
            continue
        j = np.where(unique == ouais)[0]
        height_ratios[i] = count[j]
        unique_in_order.append(ouais)
        i = i + 1
    where_lines = np.cumsum(height_ratios[::-1])
    letters = list(ascii_uppercase[:n_cbars])
    add_to_zeroone = np.zeros(n_cbars)
    # add_to_zeroone[0] = 2 if jet == "STJ" else 1
    add_to_zeroone[0] = 1
    height_ratios_ = height_ratios + add_to_zeroone
    X = df[list(relevant_keys)]
    # X_pvals = df[pvals]
    
    fig, axes = plt.subplot_mosaic([["X", letter] for letter, _ in zip(letters[::-1], height_ratios_)], width_ratios=[1, 0.017], height_ratios=height_ratios_, figsize=size, constrained_layout=True, per_subplot_kw={"X": {"aspect": "equal"}})
    
    fig.get_layout_engine().set(w_pad=0.00, h_pad=0.001, hspace=0, wspace=0)
    x = np.arange(X.shape[0] + 1)
    ys = [np.arange(height_ratios[i] + 1) for i in range(n_cbars - 1, -1, -1)]
    ys = [(y + np.sum([y[-1] for y in ys[:i]])).astype(int) for i, y in enumerate(ys)] 
    n_spells = daily_spells_list[jet]["spell"].n_unique()
    
    spell_labels = []

    spells = daily_spells_list[f"{jet}"]
    
    order_x = np.append(order_x, np.arange(3) + n_spells)
    order_x_ = order_x[:-1]
    
    spell_labels_colors = []
    for ispell in order_x_[:-2].tolist():
        spell = spells.filter(pl.col("spell") == ispell)
        year = str(spell["time"].dt.year().mode().item()).zfill(4)
        first_date, last_date = spell["time"].dt.strftime("%d/%m").gather([0, -1]).to_list()
        spell_labels.append(r"$\mathbf{" + f"{year}, {first_date}-{last_date}" + r"}$")
        
        group = groups[ispell, "group"]
        spell_labels_colors.append(colors_groups[group])
        
        # if jet == "STJ" and spells_in_early_or_late[jet]["is_early"][ispell]:
        #     spell_labels_colors.append(COLORS_EXT[-2])
        # else:
        #     spell_labels_colors.append("black")
    spell_labels.append("Episode mean") 
    spell_labels.append("Mean clim") 
    spell_labels_colors.extend(["black", "black"])
    
    for i, (unique_, otherax) in enumerate(zip(unique_in_order[::-1], letters)):
        kwargs_ = plot_kwargs[unique_].copy()
        cbar_ticks = kwargs_.pop("cbar_ticks")
        im = axes["X"].pcolormesh(x, ys[i], X.to_numpy().T[ys[i][:-1], :][:, order_x_], **kwargs_, linewidth=0.5, edgecolor="white")
        # dummy = signif_array(X_pvals, key, alpha=0.1)
        # axes["X"].pcolor(x, ys[i], dummy, hatch="////", facecolor="none", edgecolor="lightgrey", hatch_linewidth=1, linewidth=0, zorder=12)
        cbar = fig.colorbar(im, cax=axes[otherax], spacing="proportional", extendfrac="auto", pad=0.01)
        cbar.ax.set_yticks(cbar_ticks)
        
    """WRS"""
    regimes = df[order_x_, f"regime_during"].fill_null(0).cast(pl.Int32()).to_numpy()
    regime_names = winner_names["name"][regimes].to_numpy()
    cmap = colormaps.bold
    cmap.set_under("none")
    cmap.set_bad("none")
    axes["X"].pcolormesh(x, height_ratios.sum() + [0, 1], regimes[None, :], cmap=cmap, norm=BoundaryNorm(np.arange(cmap.N) + 0.5, cmap.N), linewidth=0.5, edgecolor="white")
    for x_, text_ in zip(x[:-1] + 0.5, regime_names):
        if text_ == "No":
            text_ = ""
        else:
            text_ = r"\textbf{" + text_ + r"}"
        axes["X"].text(x_, height_ratios.sum() + 0.44, text_, ha="center", va="center", fontsize=8, color="white" if text_ == "GB" else "black")
    labels = [a[1] for a in relevant_keys.values()]
    labels.append("Weather Regime")
    
    # """MJO"""
    # if jet == "STJ":
    #     phases = df[f"mjo_phase_{when}"].fill_null(0).cast(pl.Int32()).to_numpy()
    #     cmap = colormaps.pastel
    #     cmap.set_under("none")
    #     axes["X"].pcolormesh(x, height_ratios.sum() + [1, 2], phases[None, :], cmap=cmap, norm=BoundaryNorm(np.arange(cmap.N) + 0.5, cmap.N), linewidth=0.5, edgecolor="white")
    #     for x_, text_ in zip(x[:-1] + 0.5, phases.astype(str)):
    #         if text_ == "0":
    #             text_ = ""
    #         axes["X"].text(x_, height_ratios.sum() + 1.45, text_, ha="center", va="center")
    #     labels.append("MJO phase")
    
    axes["X"].set_yticks(np.arange(len(labels)) + 0.5, labels=labels)
    
    labels = axes["X"].get_yticklabels()
    for key, label in zip(relevant_keys, labels):
        if key.split("_")[-1] == "before":
            label.set_color('darkslategrey')
    for ypos in where_lines:
        axes["X"].axhline(ypos, color="black")

        
    axes["X"].set_xticks(np.arange(len(spell_labels)) + 0.5, labels=spell_labels, rotation=90)
    for color, ticklabel in zip(spell_labels_colors, axes["X"].xaxis.get_ticklabels()):
        ticklabel.set_color(color)
        ticklabel.set_color(color)
    axes["X"].set_xlabel(f"Persistent episode of the {jet}")
    
    plt.draw()
    tb = fig.get_tightbbox(fig.canvas.get_renderer())
    fig.set_size_inches(tb.width, tb.height)
    plt.draw()
    
    figW, figH = fig.get_size_inches()
    x0, _, w, _ = axes["X"].get_position().bounds
    end_of_X = x0 + w
    start_of_a = axes["A"].get_position().bounds[0]
    extra_padding = (start_of_a - end_of_X)
    new_W = figW * (1 - extra_padding)
    fig.set_size_inches(new_W, figH)
    fig.savefig(f"{FIGURES}/jet_persistence/last_v5_{jet}_{n_spells}spells.png")
    plt.plot()

In [ ]:
np.corrcoef(everything_edj[:30, "EKE250_core_during"], everything_edj[:30, "lat_over_europe_during"])

# interp

In [ ]:
varnames_rwb = ["APVO", "CPVO"]

long_names = {
    "t2m": "2m temperature",
    "tp": "Daily accum. precip.",
    "PV330": "PV@330K",
    "PV350": "PV@350K",
    "theta": r"$\theta$@2PVU",
    "EKE250": "EKE@250 hPa",
    "EMF250": "EMF@250 hPa",
    "EMFconv250": "EMF conv. @250 hPa",
    "u": "u",
    "v": "v",
    "s": "U",
    "s250": "U@250 hPa",
    "vort": "Rel. Vort.",
} | {key: f"{key[0]}WB freq" for key in varnames_rwb}

units = {
    "t2m": "K",
    "t_up": "K",
    "tp": "mm",
    "PV330": "PVU",
    "PV350": "PVU",
    "theta": "K",
    "EKE250": r"$\mathrm{m}^{2}\mathrm{s}^{-2}$",
    "EMF250": r"$\mathrm{m}^{2}\mathrm{s}^{-2}$",
    "EMFconv250": r"$\mathrm{m}\mathrm{s}^{-2}$",
    "u": r"$\mathrm{m}\mathrm{s}^{-1}$",
    "v": r"$\mathrm{m}\mathrm{s}^{-1}$",
    "s": r"$\mathrm{m}\mathrm{s}^{-1}$",
    "s250": r"$\mathrm{m}\mathrm{s}^{-1}$",
    "vort": r"$\mathrm{s}^{-1}$",
} | {varname: r"$\%$" for varname in varnames_rwb}

factors = {
    "PV": 1e6,
    "tp": 1000,
    "EMFconv250": 1e6,
    "vort": 1e6,
} | {varname: 100 for varname in varnames_rwb}

all_plot_kwargs = {
    "theta:clim": [8, colormaps.bilbao_r, [330, 370]],
    "theta:anom": [8, colormaps.BlWhRe, [-6, 6]],
    "theta:clim_grad": [8, colormaps.bilbao_r, [0, 40]],
    "theta:anom_grad": [10, colormaps.BlWhRe, [-12, 12]],
    "PV:clim": [9, WERNLI_FLAIR, [0, 10]],
    "PV:anom": [8, colormaps.BlWhRe, [-1.2, 1.2]],
    # "PV:clim_grad": [10, colormaps.bilbao_r, [0, 10]],
    # "PV:anom_grad": [10, colormaps.BlWhRe, [-4, 4]],
    "EKE250:clim": [9, colormaps.cet_l_bmy_r, [0, 200]],
    "EKE250:anom": [9, colormaps.BlWhRe, [-30, 30]],
    # "EMFconv250:clim": [12, colormaps.BlWhRe, [-120, 120]],
    # "EMFconv250:anom": [12, colormaps.BlWhRe, [-80, 80]],
    "t2m:clim": [6, colormaps.bilbao_r, [280, 300]],
    "t2m:anom": [8, colormaps.BlWhRe, [-4, 4]],
    "tp:clim": [9, colormaps.freeze_r, [0, 6]],
    "tp:anom": [9, colormaps.brbg, [-2, 2]],
    # "u:clim": [6, colormaps.bilbao_r, [0, 30]],
    # "u:anom": [8, colormaps.BlWhRe, [-12, 12]],
    # "vort:clim": [9, colormaps.BlWhRe, [-60, 60]],
    # "vort:anom": [9, colormaps.BlWhRe, [-20, 20]],
    # "v:clim": [6, colormaps.BlWhRe, [-10, 10]],
    # "v:anom": [8, colormaps.BlWhRe, [-4, 4]],
    # "s:clim": [6, colormaps.bilbao_r, [0, 40]],
    # "s:anom": [8, colormaps.BlWhRe, [-12, 12]],
    # "s250:clim": [6, colormaps.bilbao_r, [0, 40]],
    # "s250:anom": [8, colormaps.BlWhRe, [-12, 12]],
}

for varname in varnames_rwb:
    all_plot_kwargs[f"{varname}:clim"] = [7, colormaps.cet_l_bmy_r, [0, 70]]
    all_plot_kwargs[f"{varname}:anom"] = [8, colormaps.BlWhRe, [-16, 16]]
    
variables = list(all_plot_kwargs)
variables.extend(["up250:clim", "vp250:clim", "up250:anom", "vp250:anom"])
    
summer_doy = summer_daily.dt.ordinal_day().unique()

In [ ]:
def do_one(varname, basepath, jet, spells, summer, n_bootstraps, factor):
    varname, mode = varname.split(":")
    varname_ = f"{varname}_interp"
    grad = mode[-4:] == "grad"
    is_polar = jet == "EDJ"
    jet_id = int(is_polar)
    
    df = pl.scan_parquet(basepath.joinpath(f"{varname}_meters_relative.parquet"))
    if "jet ID" not in df.columns:
        df = df.with_columns(**{"jet ID": pl.col("is_polar").cast(pl.UInt32())})
    grad_expr = ((central_diff(pl.col(varname_).sort_by("n")) / central_diff(pl.col("n").sort())) * 1e6).abs()
    if grad:
        df = df.with_columns(**{varname_: grad_expr.over("norm_index", "time", "jet ID")})
    clim = compute_relative_clim(df, varname)
    if mode in ["clim", "clim_grad"]:
        to_plot = compute_relative_sm(clim, varname, summer_doy)
        to_plot = to_plot.filter(pl.col("dayofyear") == 1, pl.col("jet ID") == jet_id)
        to_plot = to_plot.drop("jet ID", "dayofyear")
        pvals = None
    else:
        to_plot = compute_relative_anom(df, varname, clim)
        ts_bootstrapped = create_bootstrapped_times(spells, summer, n_bootstraps).lazy()
        to_plot = ts_bootstrapped.join(to_plot, on="time").filter(pl.col("jet ID") == jet_id).sort("sample_index", "spell", "inside_index", "norm_index", "n")
        to_plot = to_plot.group_by("sample_index", "norm_index", "n", maintain_order=True).agg(pl.col(varname_).mean())
        pvals = to_plot.group_by("norm_index", "n", maintain_order=True).agg((pl.col(varname_).rank().last() - 1) / n_bootstraps).sort("norm_index", "n")
        pvals = pvals.with_columns(**{varname_: 2 * pl.min_horizontal(pl.col(varname_), 1 - pl.col(varname_))})
        to_plot = to_plot.filter(pl.col("sample_index") == n_bootstraps).drop("sample_index").sort("norm_index", "n")
    to_plot = to_plot.with_columns(pl.col(varname_) * factor)
    to_plot = to_plot.collect()
    to_plot = polars_to_xarray(to_plot, ["norm_index", "n"]).T
    if pvals is not None:
        pvals = pvals.collect()
        pvals = polars_to_xarray(pvals, ["norm_index", "n"]).T
    return to_plot, pvals

during_filter = [
    pl.col("relative_time") >= pl.duration(days=0), 
    pl.col("relative_time") <= pl.duration(days=3)
]
n_bootstraps = 100
for jet in ["STJ", "EDJ"]:
    spells = daily_spells_list[jet]
    nspells = spells["spell"].n_unique()
    spells = spells.filter(*during_filter)
    for varname in tqdm(variables):
        varname_, rest = varname.split(":")
        factor = factors.get(varname_, 1)
        if varname_ == "PV":
            varname = f"PV330:{rest}" if jet == "EDJ" else f"PV350:{rest}"
        ofile = Path(f"{FIGURES}/jet_persistence/figure_data/{jet}_{varname}_meters.nc")
        ofile_pvals = Path(f"{FIGURES}/jet_persistence/figure_data/{jet}_pvals_{varname}_meters.nc")
        if ofile.is_file():
            continue
        to_plot, pvals = do_one(varname, basepath, jet, spells, summer, n_bootstraps, factor)
        to_plot.to_netcdf(ofile)
        if pvals is not None:
            pvals.to_netcdf(ofile_pvals)

In [ ]:
square_len = 3
n_col = 4
pad = 0.02
fraction = 0.12
n_row = int(np.ceil(len(all_plot_kwargs) / n_col))
width = square_len * n_col * (1 + pad + fraction)
height = square_len * n_row
cbar_kwargs = dict(pad=pad, fraction=fraction, spacing="proportional")

alpha = 0.1
n_bootstraps = 100
for jet in ["STJ", "EDJ"]:
    spells = daily_spells_list[jet]
    nspells = spells["spell"].n_unique()
    fig, axes = plt.subplots(n_row, n_col, figsize=(width, height), sharex="all", sharey="all", constrained_layout=True)
    axes = axes.ravel()
    for i, ax, varname_full in zip(range(1000), axes, all_plot_kwargs):
        letter = ascii_lowercase[i % 26]
        props = all_plot_kwargs[varname_full]
        nlevels, cmap, (min_, max_) = props
        varname, mode = varname_full.split(":")
        if varname == "PV":
            varname = "PV330" if jet == "EDJ" else "PV350"
            varname_full = f"{varname}:{mode}"
        # if varname != "EKE250":
        #     continue
        
        varname_ = f"{varname}_interp"
        long_name = long_names[varname]
        grad = mode[-4:] == "grad"
        is_polar = jet == "EDJ"
        if varname[:2] == "PV" and mode == "clim":
            levels = WERNLI_FLAIR_LEVELS
            cbar_kwargs["spacing"] = "uniform"
        else:
            levels = MaxNLocator(nlevels).tick_values(min_, max_)
            cbar_kwargs["spacing"] = "proportional"
        if (-1 in np.sign(levels)) and (1 in np.sign(levels)) and varname_full != "PV330:clim" and varname_full != "PV350:clim":
            levels = np.delete(levels, np.where(levels==0)[0])
        norm = BoundaryNorm(levels, cmap.N, extend="neither")
        
        ofile = Path(f"{FIGURES}/jet_persistence/figure_data/{jet}_{varname_full}_meters.nc")
        to_plot = xr.open_dataarray(ofile)
        y = to_plot.n / 1e6
        im = ax.pcolormesh(to_plot.norm_index, y, to_plot.values, shading="nearest", cmap=cmap, norm=norm)
        
        if varname_full == "EKE250:clim":
            ofile = Path(f"{FIGURES}/jet_persistence/figure_data/{jet}_up250:{mode}_meters.nc")
            up = xr.open_dataarray(ofile).coarsen({"n": 2, "norm_index": 2}, boundary="pad").mean()
            ofile = Path(f"{FIGURES}/jet_persistence/figure_data/{jet}_vp250:{mode}_meters.nc")
            vp = xr.open_dataarray(ofile).coarsen({"n": 2, "norm_index": 2}, boundary="pad").mean()            
            C = np.abs(to_plot[::2, ::2]) > levels[-3]
            cmap_ = colormaps.gray
            norm_ = BoundaryNorm([0, 0.2, 1], cmap_.N)
            ax.quiver(up.norm_index, up.n / 1e6, up.values, vp.values, C, norm=norm_, cmap=cmap_, width=.005, headwidth=3, pivot="mid", scale=60)
        
        ofile_pvals = Path(f"{FIGURES}/jet_persistence/figure_data/{jet}_pvals_{varname_full}_meters.nc")
        if ofile_pvals.is_file() and varname_full != "EKE250:clim":
            pvals = xr.open_dataarray(ofile_pvals)
            pvals_ = pvals.values.ravel()
            sort_order = np.argsort(pvals_)
            try:
                cutoff = np.amax(np.where(pvals_[sort_order] <= np.linspace(0, 1, len(sort_order)) * alpha)[0])
            except ValueError:
                cutoff = 0
            filter_ = np.argsort(sort_order).reshape(pvals.shape) >= cutoff
            crrr = ax.pcolor(to_plot.norm_index, y, pvals.where(~filter_), hatch="///", facecolor="none", edgecolor="gray", hatch_linewidth=2, linewidth=0)
        ax.axhline(0, lw=2.5, color=COLORS[2 - int(is_polar)])
        ax.set_xlim(0, 1)
        unit = units[varname] + r"$/10^{6}\mathrm{m}$" if grad else units[varname]
        unit = "pp" if unit == r"$\%$" and mode == "anom" else unit
        mode_pretty = mode.split("_")
        mode_pretty = f"{mode_pretty[0]} of {mode_pretty[1]}" if "_" in mode else mode
        ax.set_title(f"{letter}) {long_name}, {mode_pretty} [{unit}]", fontsize=13)
        cbar = fig.colorbar(im, ax=ax, **cbar_kwargs)
        cbar.ax.yaxis.set_tick_params(labelsize=13)
        if i >= (n_row - 1) * n_col:
            ax.set_xlabel("Along jet coord.")
        if i % n_col == 0:
            ax.set_ylabel(r"Normal distance $[10^{6} \mathrm{m}]$")
            
    fig.savefig(f"/Users/bandelol/Documents/code_local/local_figs/jet_persistence/{jet}_interp_{nspells}spells_v2.png")

### extended

In [ ]:
all_plot_kwargs = {
    "PV:clim": [9, WERNLI_FLAIR, [0, 10]],
    "PV:anom": [8, colormaps.BlWhRe, [-1.2, 1.2]],
    "PV:clim_grad": [10, colormaps.bilbao_r, [0, 10]],
    "PV:anom_grad": [10, colormaps.BlWhRe, [-4, 4]],
    "EKE250:clim": [9, colormaps.cet_l_bmy_r, [0, 200]],
    "EKE250:anom": [9, colormaps.BlWhRe, [-30, 30]],
    "EMFconv250:clim": [12, colormaps.BlWhRe, [-120, 120]],
    "EMFconv250:anom": [12, colormaps.BlWhRe, [-80, 80]],
    # "u:clim": [6, colormaps.bilbao_r, [0, 30]],
    # "u:anom": [8, colormaps.BlWhRe, [-12, 12]],
    "vort:clim": [9, colormaps.BlWhRe, [-60, 60]],
    "vort:anom": [9, colormaps.BlWhRe, [-20, 20]],
    "v:clim": [6, colormaps.BlWhRe, [-10, 10]],
    "v:anom": [8, colormaps.BlWhRe, [-4, 4]],
    "s:clim": [6, colormaps.bilbao_r, [0, 40]],
    "s:anom": [8, colormaps.BlWhRe, [-12, 12]],
    "s250:clim": [6, colormaps.bilbao_r, [0, 40]],
    "s250:anom": [8, colormaps.BlWhRe, [-12, 12]],
}

# for varname in varnames_rwb:
#     all_plot_kwargs[f"{varname}:clim"] = [7, colormaps.cet_l_bmy_r, [0, 70]]
#     all_plot_kwargs[f"{varname}:anom"] = [8, colormaps.BlWhRe, [-16, 16]]
    
variables = list(all_plot_kwargs)
variables.extend(["up250:clim", "vp250:clim", "up250:anom", "vp250:anom"])

square_len = 3
n_col = 4
pad = 0.02
fraction = 0.12
n_row = int(np.ceil(len(all_plot_kwargs) / n_col))
width = square_len * n_col * (1 + pad + fraction)
height = square_len * n_row
cbar_kwargs = dict(pad=pad, fraction=fraction, spacing="proportional")

alpha = 0.1
n_bootstraps = 100
for jet in ["STJ", "EDJ"]:
    spells = daily_spells_list[jet]
    nspells = spells["spell"].n_unique()
    fig, axes = plt.subplots(n_row, n_col, figsize=(width, height), sharex="all", sharey="all", constrained_layout=True)
    axes = axes.ravel()
    for i, ax, varname_full in zip(range(1000), axes, all_plot_kwargs):
        letter = ascii_lowercase[i % 26]
        props = all_plot_kwargs[varname_full]
        nlevels, cmap, (min_, max_) = props
        varname, mode = varname_full.split(":")
        if varname == "PV":
            varname = "PV330" if jet == "EDJ" else "PV350"
            varname_full = f"{varname}:{mode}"
        # if varname != "EKE250":
        #     continue
        
        varname_ = f"{varname}_interp"
        long_name = long_names[varname]
        grad = mode[-4:] == "grad"
        is_polar = jet == "EDJ"
        if varname[:2] == "PV" and mode == "clim":
            levels = WERNLI_FLAIR_LEVELS
            cbar_kwargs["spacing"] = "uniform"
        else:
            levels = MaxNLocator(nlevels).tick_values(min_, max_)
            cbar_kwargs["spacing"] = "proportional"
        if (-1 in np.sign(levels)) and (1 in np.sign(levels)) and varname_full != "PV330:clim" and varname_full != "PV350:clim":
            levels = np.delete(levels, np.where(levels==0)[0])
        norm = BoundaryNorm(levels, cmap.N, extend="neither")
        
        ofile = Path(f"{FIGURES}/jet_persistence/figure_data/{jet}_{varname_full}_meters.nc")
        to_plot = xr.open_dataarray(ofile)
        y = to_plot.n / 1e6
        im = ax.pcolormesh(to_plot.norm_index, y, to_plot.values, shading="nearest", cmap=cmap, norm=norm)
        
        if varname_full == "EKE250:clim":
            ofile = Path(f"{FIGURES}/jet_persistence/figure_data/{jet}_up250:{mode}_meters.nc")
            up = xr.open_dataarray(ofile).coarsen({"n": 2, "norm_index": 2}, boundary="pad").mean()
            ofile = Path(f"{FIGURES}/jet_persistence/figure_data/{jet}_vp250:{mode}_meters.nc")
            vp = xr.open_dataarray(ofile).coarsen({"n": 2, "norm_index": 2}, boundary="pad").mean()            
            C = np.abs(to_plot[::2, ::2]) > levels[-3]
            cmap_ = colormaps.gray
            norm_ = BoundaryNorm([0, 0.2, 1], cmap_.N)
            ax.quiver(up.norm_index, up.n / 1e6, up.values, vp.values, C, norm=norm_, cmap=cmap_, width=.005, headwidth=3, pivot="mid", scale=60)
            
        if varname in ["s", "s250"]:
            tplt = to_plot.values
            centroid = np.sum(to_plot["n"].values[:, None] * (tplt - tplt.min()), axis=0) / (tplt - tplt.min()).sum(axis=0)
            ax.plot(to_plot["norm_index"].values, centroid / 1e6, lw=2.5, color="lime")
            argmax = to_plot["n"].values[tplt.argmax(axis=0)]
            ax.plot(to_plot["norm_index"].values, argmax / 1e6, lw=2.5, color="cyan")
                  
        ofile_pvals = Path(f"{FIGURES}/jet_persistence/figure_data/{jet}_pvals_{varname_full}_meters.nc")
        if ofile_pvals.is_file() and varname_full != "EKE250:clim":
            pvals = xr.open_dataarray(ofile_pvals)
            pvals_ = pvals.values.ravel()
            sort_order = np.argsort(pvals_)
            try:
                cutoff = np.amax(np.where(pvals_[sort_order] <= np.linspace(0, 1, len(sort_order)) * alpha)[0])
            except ValueError:
                cutoff = 0
            filter_ = np.argsort(sort_order).reshape(pvals.shape) >= cutoff
            crrr = ax.pcolor(to_plot.norm_index, y, pvals.where(~filter_), hatch="///", facecolor="none", edgecolor="gray", hatch_linewidth=2, linewidth=0)
        ax.axhline(0, lw=2.5, color=COLORS[2 - int(is_polar)])
        ax.set_xlim(0, 1)
        unit = units[varname] + r"$/10^{6}\mathrm{m}$" if grad else units[varname]
        unit = "pp" if unit == r"$\%$" and mode == "anom" else unit
        mode_pretty = mode.split("_")
        mode_pretty = f"{mode_pretty[0]} of {mode_pretty[1]}" if "_" in mode else mode
        ax.set_title(f"{letter}) {long_name}, {mode_pretty} [{unit}]", fontsize=13)
        cbar = fig.colorbar(im, ax=ax, **cbar_kwargs)
        cbar.ax.yaxis.set_tick_params(labelsize=13)
        if i >= (n_row - 1) * n_col:
            ax.set_xlabel("Along jet coord.")
        if i % n_col == 0:
            ax.set_ylabel(r"Normal distance $[10^{6} \mathrm{m}]$")
            
    fig.savefig(f"/Users/bandelol/Documents/code_local/local_figs/jet_persistence/{jet}_interp_{nspells}spells_appendix.png")

# riviere 2009

In [ ]:
import numpy as np
from jetutils.definitions import degcos, degsin
import matplotlib.pyplot as plt
phi = np.radians(np.linspace(0, 90, 100))
phi0 = np.radians(35) 
phiv = np.radians(15) #
u0 = 50 # m.s-1
deltau0 = 10 # m
R_0 = 660_000 # m
gphi = -((phi - phi0) / 2 / phiv) ** 2 # no
u = u0 * np.exp(gphi) # m / s
RADIUS = 6.371e6  # m
OMEGA = 7.2921e-5  # rad.s-1
VELOCITY = 3e-6 # s-1
f = 2 * OMEGA * degsin(phi) # rad.s-1
f_0 = 2 * OMEGA * degsin(45)  # rad.s-1
R_1 = R_0 * f_0 / f # m
zeta_phi = - u * ((phi - phi0) ** 2 - phiv ** 2) / phiv ** 4 # 
q_y = 2 * OMEGA * degcos(phi) / RADIUS + zeta_phi / RADIUS + R_1 ** (-2) * deltau0 * np.exp(gphi)
plt.plot(np.degrees(phi), q_y / 6.5e-4, lw=3)
plt.plot(np.degrees(phi), (u - VELOCITY * RADIUS * degcos(phi)) / 32, lw=1)
# plt.plot(2 * OMEGA * degcos(phi) / RADIUS / 6.5e-4, marker="x")
# plt.plot(zeta_phi / RADIUS, marker="s")

# anim

In [ ]:
import altair as alt
spell_jets = extend_spells(spells_list["STJ"], time_before=datetime.timedelta(days=6), time_after=datetime.timedelta(days=1)).join(phat_jets, on="time")
slider_spell = alt.binding_range(min=spell_jets["spell"].min(), max=spell_jets["spell"].max(), step=1)
spell_value = alt.param("spell", bind=slider_spell, value=0)

slider_rel_time = alt.binding_range(min=spell_jets["relative_index"].min(), max=spell_jets["relative_index"].max(), step=1)
rel_time_value = alt.param("relative_index", bind=slider_rel_time, value=0)

a = (
    alt
    .Chart(spell_jets)
    .mark_point()
    .encode(
        x=alt.X("lon:Q").scale(domain=[-80, 40]),
        y=alt.Y("lat:Q").scale(domain=[20, 80]),
    )
    .transform_filter(
        alt.datum["spell"] == spell_value,
        alt.datum["relative_index"] == rel_time_value,
    )
    .add_params(spell_value, rel_time_value)
)
a

# tests

In [ ]:
import datetime
cross_catd = pl.read_parquet(cross_catd_ofile)
cross_catd = pers_from_cross_catd(cross_catd)
cross_catd = squarify(cross_catd, ["time", "spell_of"])
cross_catd = cross_catd.with_columns(**{"jet ID": (pl.col("spell_of") == "EDJ").cast(pl.UInt32())})
smooth = datetime.timedelta(hours=48)
fill_holes = datetime.timedelta(hours=12)
base_q = 0.6
minlen = datetime.timedelta(days=5)
if smooth is not None:
    cross = cross_catd.rolling(
        pl.col("time"),
        period=smooth,
        group_by=["jet ID", "spell_of"],
    ).agg(*[pl.col(col).mean() for col in ["lon_overlap", "ds", "dtheta", "dis_polar", "dist", "pers"]])
else:
    cross = cross_catd.clone()

season = summer
if season is not None:
    cross = season.rename("time").to_frame().join(cross, on="time", how="left")
    
spells_base = get_spells(cross, pl.col("pers") > pl.col("pers").quantile(base_q), group_by=["spell_of"], minlen=minlen, fill_holes=fill_holes)
    
spells_list = spells_from_cross_catd(pl.read_parquet(cross_catd_ofile), season=summer, base_q=base_q, n_STJ=30, n_EDJ=30, minlen=minlen, smooth=smooth, fill_holes=fill_holes)

In [ ]:
spell_of = "STJ"
pers_edj_summer = cross.filter(pl.col("spell_of") == spell_of)
q06 = pers_edj_summer.select(pl.col("pers").quantile(base_q)).item()
spells = spells_list[spell_of]
n_spell = np.random.choice(spells["spell"].unique())
# n_spell = 18
spell = spells_list[spell_of].filter(pl.col("spell") == n_spell)
spell_ext = extend_spells(spells_list[spell_of], time_before=datetime.timedelta(days=12), time_after=datetime.timedelta(days=12)).filter(pl.col("spell") == n_spell).join(pers_edj_summer[["time", "spell_of", "pers"]], on=["time", "spell_of"])

# spell_ext_raw = extend_spells(spells_list[spell_of], time_before=datetime.timedelta(days=12), time_after=datetime.timedelta(days=12)).filter(pl.col("spell") == n_spell).drop("pers").join(cross_catd.filter(pl.col("spell_of") == spell_of)[["time", "pers"]], on="time", how="left")

plt.plot(spell_ext["relative_index"], spell_ext["pers"])
plt.plot(spell_ext["relative_index"], spell_ext["pers_right"])
# plt.plot(spell_ext_raw["relative_index"], spell_ext_raw["pers"].is_null() * 0.1)

plt.plot(spell_ext["relative_index"][[0, -1]], [q06, q06], zorder=-30)
plt.axvline(0, color="red", ls="dashed", zorder=-30)
plt.axvline(spell["relative_index"].max(), color="red", ls="dashed", zorder=-30)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, rgb_to_hsv, hsv_to_rgb
from tqdm import trange
from jetutils.jet_finding import extract_features
from jetutils.definitions import MONTH_NAMES

xys = []
all_jets = all_jets_one_df

xys = np.array(xys)
fig, axes = plt.subplots(3, 4, figsize=(11, 9), tight_layout=True, sharex="all", sharey="all")
axes = axes.ravel()
pair = ["s", "theta", "is_polar"]
cmap = LinearSegmentedColormap.from_list("pinkwhitepurple", [COLORS[2], "#ffffff", COLORS[1]])
bins = np.linspace(0, 1, 31)
for month in trange(1, 13):
    ax = axes[month - 1]
    X = extract_features(all_jets, pair, season=month)
    probas = X[pair[2]]
    center_stj = X.filter(pl.col("is_polar") < 0.3).mean()
    center_edj = X.filter(pl.col("is_polar") > 0.7).mean()
    X1D = X["is_polar"]
    
    im1 = ax.hexbin(X[pair[0]], X[pair[1]], cmap=colormaps.gray_r, gridsize=25)
    im2 = ax.hexbin(X[pair[0]], X[pair[1]], C=probas, cmap=colormaps.gray_r, gridsize=25)
    
    plt.draw()
        
    offsets1 = np.asarray(list(map(tuple, im1.get_offsets())), dtype="f, f")
    offsets2 = np.asarray(list(map(tuple, im2.get_offsets())), dtype="f, f")
    mask12 = np.isin(offsets1, offsets2)
    colors = cmap(im2.get_array())
    colors = rgb_to_hsv(colors[:, :3])
    min_s, max_s = 0., 1.0
    min_v, max_v = 0.75, 1.
    scaling = np.clip(im1.get_array()[mask12] / im1.get_array()[mask12].max() * 1.1, 0, 1)
    f = lambda x: np.sqrt(x)
    colors[:, 1] = min_s + scaling * (max_s - min_s)
    colors[:, 2] = max_v - scaling * (max_v - min_v)
    colors = hsv_to_rgb(colors)
    im2.set_array(None)
    im2.set_facecolor(colors)
    # im2.set_linewidths(0.2)
    im2.set_linewidths(4 * (1 - f(scaling)) ** 2)
    im2.set_edgecolor(colormaps.greys(scaling))
    im2 = ax.add_collection(im2)
        
    ax.set_xlabel(pair[0])
    ax.set_ylabel(pair[1])
    if pair[0] in ["lev", "theta"]:
        ax.invert_xaxis()
    elif pair[1] in ["lev", "theta"]:
        ax.invert_yaxis() 
        
    ax.set_title(MONTH_NAMES[month - 1])
    ax.scatter(*pl.concat([center_stj, center_edj])[pair[:2]].to_numpy().T, facecolor="black", edgecolor="white", marker="X", linewidths=1, s=45)
    iax = ax.inset_axes([0.65, 0.2, 0.4, 0.5])
    X1D = np.clip(X1D, 0, 1)
    iax.hist(X1D, bins=bins, alpha=0.5, color="black")
    iax.hist(X1D[probas > 0.5], bins=bins, alpha=0.5, color=COLORS[1])
    iax.hist(X1D[probas < 0.5], bins=bins, alpha=0.5, color=COLORS[2])
    iax.set_xticks([0, 0.5, 1])
    iax.set_yticks([])
    iax.spines[["left", "right", "top"]].set_visible(False)
    plt.draw()


# when spells

### full grid

In [ ]:
plt.style.use(['seaborn-v0_8-darkgrid', STYLE_SHEET])
colors = [COLORS[2], COLORS[1]]

fig, ax = plt.subplots(figsize=(1, 0.5), dpi=100)
ax.axis("off")
for spell_name, color in zip(spells_list, colors):
    ax.plot([], [], color=color, lw=3, label=spell_name)
ax.legend()
plt.show()
min_summer, max_summer =  summer.dt.ordinal_day().unique().first(),  summer.dt.ordinal_day().unique().last()
fig, axes = plt.subplots(
    8,
    8,
    figsize=(9, 4),
    gridspec_kw=dict(wspace=.05, hspace=.05, left=0, right=1, bottom=0, top=1),
    subplot_kw=dict(xticks=[], yticks=[], xlim=[-1, max_summer - min_summer], ylim=[-1, len(colors) + 1]),
)
axes = axes.ravel()
for ax, year in zip(axes, YEARS):
    ax.text(70, len(spells_list) - 0.3, f"{year}", fontsize=10)
    for j, (name_, spell) in enumerate(spells_list.items()):
        ax.plot([-1, max_summer - min_summer], [j, j], color=colors[j], lw=0.5, ls="solid", alpha=0.5)
        spell_ = spell.filter(pl.col("time").dt.year() == year)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=colors[j], lw=3)
plt.show()
plt.style.use(["default", STYLE_SHEET])
fig.savefig(f"{FIGURES}/jet_persistence/when_jets.png")

In [ ]:
from matplotlib.dates import DateFormatter

fig, axes = plt.subplots(1, 3, figsize=(10, 4), gridspec_kw=dict(wspace=0, hspace=0))

ax = axes[0]
colors = [COLORS[2], COLORS[1]]
keys = list(spells_list)
i = 1
otheralpha = 0.5
dy = 30
for key, val in spells_list.items():
    huh = (
        summer.dt.ordinal_day()
        .unique()
        .sort()
        .to_frame()
        .with_columns(time_2=pl.datetime(year=1959, month=1, day=1) + pl.duration(days="time"))
    )
    to_plot = gaussian_kde(val["time"].dt.ordinal_day(), bw_method=0.08).evaluate(huh["time"])
    ax.fill_between(
        huh["time_2"],
        i,
        i + dy * to_plot,
        color=colors[1 - i],
        facecolor="none",
        alpha=1.0,
        lw=2
    )
    ax.fill_between(
        huh["time_2"],
        i,
        i + dy * to_plot,
        color=colors[1 - i],
        alpha=otheralpha,
    )
    i = i - 1
ax.xaxis.set_major_formatter(DateFormatter("%m-%d"))
ax.xaxis.set_tick_params(labelsize=11, rotation=30)
ticks = ax.get_xticks()
ticklabels = ax.get_xticklabels()
ax.set_xticks(ticks, labels=[t.get_text() for t in ticklabels], ha="right")
ax.set_yticks([0.35, 1.5], labels=keys[::-1])
ax.yaxis.set_tick_params(length=0)
ax.set_title("a) Episode distribution")
ylim = ax.get_ylim()
ax.set_ylim(ylim[0], 2 - ylim[0])

ax = axes[1]
dy = 1.2
bins = np.logspace(-5.0, -0.1, 101)
summer_pers = summer_filter.join(pers, on="time")
y1 = summer_pers.filter(pl.col("jet ID") == 0)["pers"]
y2 = summer_pers.filter(pl.col("jet ID") == 1)["pers"]
for i, y in enumerate([y1, y2]): 
    y_smo = gaussian_kde(np.log10(y.replace(0, None).drop_nulls()), bw_method=0.2).evaluate(np.log10(bins))
    y_smo = (1 - i) + dy *  y_smo
    ax.fill_between(bins, (1 - i), y_smo, color=COLORS[2 - i], alpha=otheralpha)
    ax.plot(bins, y_smo, color=COLORS[2 - i], lw=2)
ax.set_xscale("log")
ax.set_yticks([])
ax.set_title("b) Persistence [s/m]")
ylim = ax.get_ylim()
ax.set_ylim(ylim[0], 2 - ylim[0])

ax = axes[2]
dy = 2
bins = np.arange(5, 20, 0.15) - 0.125
for i, jet in enumerate(["EDJ", "STJ"]): 
    y = spells_list[jet].group_by("spell").agg(pl.col("len").first())["len"] / 4
    y_smo = gaussian_kde(y, bw_method=0.2).evaluate(bins)
    y_smo = i + dy * y_smo
    ax.fill_between(bins, i, y_smo, color=COLORS[1 + i], alpha=otheralpha)
    ax.plot(bins, y_smo, color=COLORS[1 + i], lw=2)
ax.set_yticks([0, 0.5, 1, 1.5], labels=[str(i) for i in [0, 0.5] * 2])
ax.tick_params("y", left=False, right=True, labelleft=False, labelright=True)
ax.set_title("c) Episode length [day]")
ylim = ax.get_ylim()
ax.set_ylim(ylim[0], 2 - ylim[0])
fig.savefig(f"{FIGURES}/jet_persistence/occurrence_frequency_30spells.png")

### with extremes

In [ ]:
clusters_da = np.abs(xr.open_dataarray(basepath.joinpath("cluster_df.nc")).load())
clusters_da = clusters_da.interp(lat=np.arange(32, 72, 0.5), method="nearest")

# plt.savefig(f"{FIGURES}/jet_persistence/regions.png")

region_T = pl.read_parquet(basepath.joinpath("regionalized_T_anom.parquet"))
region_T = region_T.rolling("time", period="3d", group_by="region").agg(pl.col("t2m").mean())
hws = get_spells(region_T, pl.col("t2m") > pl.col("t2m").quantile(0.95), group_by=["region"], fill_holes=datetime.timedelta(hours=18), minlen=datetime.timedelta(days=3)).sort("region")
region_tp = pl.read_parquet(basepath.joinpath("regionalized_tp_anom.parquet"))
region_tp = region_tp.rolling("time", period="3d", group_by="region").agg(pl.col("tp").mean())
pes = get_spells(region_tp, pl.col("tp") > pl.col("tp").quantile(0.95), group_by=["region"], fill_holes=datetime.timedelta(hours=6), minlen=datetime.timedelta(days=3)).sort("region")
drs = get_spells(region_tp, pl.col("tp") < pl.col("tp").quantile(0.05), group_by=["region"], fill_holes=datetime.timedelta(hours=6), minlen=datetime.timedelta(days=3)).sort("region")

In [ ]:
long_names = {
    "t2m": "2m temperature [K]",
    "t_up": "Upper level temperature [K]",
    "tp": "Daily precip. [mm]",
    "apvs": r"Anticyclonic PV streamer freq [$\%$]",
    "cpvs": r"Cyclonic PV streamer freq [$\%$]",
    "theta": "Upper-level Potential temperature [K]",
    "s": r"Wind speed [$m.s^{-1}$]",
    "theta": "Pot temp at 2 PVU [K]",
    "z200": "Z200 anomaly [m]",
    "z500": "Z500 anomaly [m]",
}

In [ ]:
colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
figure = plt.figure(figsize=(10, 4.5), constrained_layout=True)
subfigs = figure.subfigures(1, 2, wspace=0.00, width_ratios=[6.5, 3.5])
with plt.style.context(['seaborn-v0_8-darkgrid', STYLE_SHEET]):
    fig = subfigs[0]
    axes = fig.subplots(2, 2, sharex="col", sharey="row")
    axes = axes.T
    for i, spell_of in enumerate(["STJ", "EDJ"]):
        spells = extend_spells(daily_spells_list[spell_of])
        for j, (name, df) in enumerate({"t2m": region_T, "tp": region_tp}.items()):
            ax = axes[i, j]
            region_ts_masked = mask_from_spells_pl(spells, df, time_before=datetime.timedelta(days=4))
            region_ts_masked = subset_around_onset(region_ts_masked, around_onset=datetime.timedelta(days=13))
            x, alive_spells = region_ts_masked.group_by(["region", "relative_index"]).agg(pl.col("spell").n_unique()).sort("relative_index").filter(pl.col("region") == 1).to_numpy().T[1:]
            filter_ = alive_spells > 3
            huh = region_ts_masked.group_by(["region", "relative_index"]).mean().sort(["region", "relative_index"])
            ax.axhline(0, color="black")
            ax.axvline(0, color="black")
            for reg, huh_ in huh.group_by("region", maintain_order=True):
                y = huh_[name].to_numpy()
                y = y * 1000 if name == "tp" else y
                ax.plot(x[filter_], y[filter_], color=colors_regions[reg[0] - 1], label=DUNCANS_REGIONS_NAMES[reg[0] - 1], lw=3)
            k = 2 * j + i
            ax.annotate(
                f"{ascii_lowercase[k]})",
                xy=(0, 1), xycoords='axes fraction',
                xytext=(+0.5, -0.5 - 6.5 * float(k==0)), textcoords='offset fontsize',
                fontsize='medium', verticalalignment='top', fontfamily='serif',
                bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0)
            )
    # axes[0, 0].legend(ncol=2, fontsize=13.5, labelspacing=0.3, markerscale=.1, columnspacing=.6, facecolor="white", framealpha=.5, fancybox=True, frameon=True)
    axes[0, 0].set_ylabel(long_names["t2m"])
    axes[0, 1].set_ylabel(long_names["tp"])
    axes[0, 0].set_title("STJ episodes")
    axes[1, 0].set_title("EDJ episodes")
    axes[0, 1].set_xlabel("Time around onset [day]")
    axes[1, 1].set_xlabel("Time around onset [day]")
    ylim = axes[0, 1].get_ylim()
    for i, spell_of in enumerate(["STJ", "EDJ"]):
        spells = extend_spells(daily_spells_list[spell_of], time_before=datetime.timedelta(days=4))
        spells = subset_around_onset(spells, around_onset=datetime.timedelta(days=13))
        x, alive_spells = spells.group_by("relative_index").agg(pl.col("spell").n_unique()).sort("relative_index").to_numpy().T
        filter_ = alive_spells > 3
        ax = axes[i, 1]
        ybounds = [ylim[0] - 0.05 * (ylim[1] - ylim[0]), ylim[0] + 0.05 * (ylim[1] - ylim[0])]
        im = ax.pcolormesh(
            x[filter_], ybounds, alive_spells[None, filter_][:, :-1], zorder=-10,
            cmap=colormaps.greys, alpha=0.7, vmin=0
        )
    axes[0, 1].set_ylim(ybounds[0])
    

clu = Clusterplot(1, 1, (-10, 40, 35, 72), row_height=5, fig=subfigs[1])
cmap = colormaps.pastel
ax = clu.axes[0]
unique_clusters = np.arange(1, 7)
norm = BoundaryNorm(np.arange(cmap.N) + 0.5, cmap.N)
clusters_da.assign_coords(lon=clusters_da.lon - clu.central_longitude).plot(
    ax=ax,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    add_labels=False
)
for j in range(6):
    lo = clusters_da.lon.where(clusters_da==(j + 1)).mean().item() - j - 3 * (j == 0) + 2 * (j == 2) + 3 * (j == 1) - (j == 4)  - clu.central_longitude
    la = clusters_da.lat.where(clusters_da==(j + 1)).mean().item() - (j == 5) * 2
    color = "black"
    ax.text(lo, la, DUNCANS_REGIONS_NAMES[j], ha="center", va="center", fontweight="bold", color=color, fontsize=12)
    
ax.annotate(
    f"{ascii_lowercase[k + 1]})",
    xy=(float(k==0), 1), xycoords='axes fraction',
    xytext=(+0.5 - 2 * float(k==0), -0.5), textcoords='offset fontsize',
    fontsize='medium', verticalalignment='top', fontfamily='serif',
    bbox=dict(facecolor='0.7', edgecolor='none', pad=3.0)
)
figure.savefig(f"{FIGURES}/jet_persistence/region_timeseries_30spells.png")

In [ ]:
from matplotlib.colors import hex2color
names = ["Cyan", "Yellow", "Orange", "Purple", "Green", "Blue"]
names = [f"{name}Pastel" for name in names]
base = r"\definecolor{"
middle = r"}{rgb}{"
end = r"}"
for color, name in zip(cmap(norm(np.arange(1, 7))), names):
    color = [f"{spec:.3f}" for spec in color[:3]]
    print(base + f"{name}" + middle + " ".join(color) + end)
    
name = "STJcolor"
color = hex2color(COLORS[2])
color = [f"{spec:.3f}" for spec in color[:3]]
print(base + f"{name}" + middle + " ".join(color) + end)

name = "EDJcolor"
color = hex2color(COLORS[1])
color = [f"{spec:.3f}" for spec in color[:3]]
print(base + f"{name}" + middle + " ".join(color) + end)

base = r"\textcolor{"
middle = r"}{\textbf{"
end = r"}}"
schema = []
for color, name in zip(names, DUNCANS_REGIONS_NAMES):
    schema.append(base + color + middle + name + end)
print(" & ".join(schema))

In [ ]:
def common_OR(i: int, spell_of: str) -> Tuple[pl.Expr]:
    spell_c = pl.col(f"reg{i}")
    jet_c = pl.col(f"spell_{spell_of}")
    yesyes = (spell_c & jet_c).sum()
    yesno = (spell_c & ~jet_c).sum()
    noyes = (~spell_c & jet_c).sum()
    nono = (~spell_c & ~jet_c).sum()
    return yesyes, nono, noyes, yesno
    
def odds_ratio(i: int, spell_of: str) -> pl.Expr:
    yesyes, nono, noyes, yesno = common_OR(i, spell_of)
    OR_ = yesyes * nono / yesno / noyes
    return OR_

def is_signif_OR(i: int, spell_of: str) -> pl.Expr:
    yesyes, nono, noyes, yesno = common_OR(i, spell_of)
    OR = yesyes * nono / yesno / noyes
    SE = (1 / yesyes + 1 / nono + 1 / yesno + 1 / noyes).sqrt()
    L = OR.log()
    one_not_in_CI = ((L - 1.96 * SE).exp() > 1) | ((L + 1.96 * SE).exp() < 1)
    return one_not_in_CI
    

for key, df in {"hw": hws, "pe": pes, "dr": drs}.items():
    out = summer_daily.to_frame()

    for region in range(1, 7):
        alias = f"reg{region}"
        to_join = df[["region", "time"]].with_columns((pl.col("region") == region).alias(alias)).filter(alias).drop("region")
        out = out.join(to_join, on="time", how="left")
    for spell_of in ["STJ", "EDJ"]:
        spells = daily_spells_list[f"{spell_of}"]
        spells = spells[["time", "spell"]].with_columns(spell=pl.lit(True)).rename({"spell": f"spell_{spell_of}"})
        out = out.join(spells, on="time", how="left")
    out = out.fill_null(False)

    aggs = {}
    for i, spell_of in product(range(1, 7), ["STJ", "EDJ"]):
        aggs[f"{i}{spell_of}"] = odds_ratio(i, spell_of)
        aggs[f"{i}{spell_of}_signif"] = is_signif_OR(i, spell_of)
    out = out.select(**aggs)

    out = out.to_numpy().reshape(6, 4)
    overlaps_ = out[:, [0, 2]].T
    pvals_ = out[:, [1, 3]].T

    hoho = (
        pl
        .DataFrame(overlaps_, schema=DUNCANS_REGIONS_NAMES)
        .with_columns([(pl.col(region)).round(1) for region in DUNCANS_REGIONS_NAMES])
        .cast({region: pl.String() for region in DUNCANS_REGIONS_NAMES})
        .with_columns(
            **{
                f"start{region}": pl.when(pl.lit(pl.Series(None, pvals_[:, i]) > 0.95)).then(pl.lit(r"$\mathbf{")).otherwise(pl.lit(r"${")) 
                for i, region in enumerate(DUNCANS_REGIONS_NAMES)
            }
        )
        .with_columns(**{region: pl.col(f"start{region}") + pl.col(region) + r"}$" for region in DUNCANS_REGIONS_NAMES})
        .drop([f"start{region}" for region in DUNCANS_REGIONS_NAMES])
    )
    print(key, hoho)
    hoho.to_pandas().to_latex(
        buf=f"OR_{key}.tex",
        escape=False,
        column_format="l",
        multirow=False,
        header=True,
        index_names=False,
    )

### wrs

In [ ]:
wr_names = ["No", "GLBl", "ScTr", "EuBl", "ScBl"]
colors = ["#6C6C6C", "#7E7EF4", "#F2A685", "#8FC386", "green"]

grams_wr = pl.read_parquet(f"{DATADIR}/grams_wr.parquet")
grams_wr = grams_wr.with_columns(**{f"winner_{i}": pl.when(pl.col("winner") == i).then(pl.lit(1.)).otherwise(pl.lit(0.)) for i in range(5)})
width = 0.25
fig, axes = plt.subplot_mosaic([['a)', 'b)', 'c)'], ['a)', 'b)', 'd)']], figsize=(8, 4), constrained_layout=True, sharey=True, width_ratios=[1, 1, 0.5], gridspec_kw=dict(wspace=0.1))
for i, (spell_of, ax) in enumerate(zip(["STJ", "EDJ"], list(axes.values()))):
    grams_wr_masked = mask_from_spells_pl(spells_list[spell_of], grams_wr, time_before=datetime.timedelta(days=3))
    huh = grams_wr_masked.group_by(["relative_index"]).mean().sort("relative_index")
    alive_spells = grams_wr_masked.group_by("relative_index").agg(pl.col("spell").n_unique()).sort("relative_index")["spell"].to_numpy()
    x = huh["relative_index"].to_numpy() / 4
    filter_ = alive_spells > 3
    x = x[filter_]
    bottom = np.zeros(len(x))
    for j in [*np.arange(1, 5), 0]:
        height = huh[f"winner_{j}"].to_numpy()[filter_]
        ax.bar(x, height, bottom=bottom, facecolor=colors[j], width=width, label=wr_names[j])    
        bottom = bottom + height
    ax.set_xlabel("Relative time around onset [day]")
    ax.set_title(f"{ascii_lowercase[i]}) {alive_spells.max():n} episodes of the {spell_of[:3]}")
    ax.set_xlim(x[0] - width / 2, x[-1] + width / 2)
    ybounds = [1, 1.05]
    im = ax.pcolormesh(
        x, ybounds, alive_spells[filter_][None, 1:], zorder=2,
        cmap=colormaps.greys, alpha=0.7, vmin=0,
    )
    ax.axhline(1, color="black")
    ax.vlines(0, 0, 1, color="black", ls="dotted", lw=1.2)
h, l = axes["b)"].get_legend_handles_labels()
axes["c)"].set_axis_off()
axes["c)"].legend(h, l, fontsize=12, ncol=1, loc="upper left", title="WR names")
axes["a)"].set_ylabel("WR frequency")
ax = axes["d)"]
monthly_means = grams_wr.filter(pl.col("time").dt.month() > 5).group_by(pl.col("time").dt.month()).agg(*[pl.col(f"winner_{i}").mean() for i in range(5)]).sort("time")
x = np.array([6, 7, 8, 9])
bottom = np.zeros(len(x))
for j in [*np.arange(1, 5), 0]:
    height = monthly_means[f"winner_{j}"].to_numpy()
    ax.bar(x, height, bottom=bottom, facecolor=colors[j], width=.9, label=wr_names[j])    
    bottom = bottom + height
ax.set_xticks(x, labels="JJAS")
ax.set_title("c) Monthly means")
fig.savefig(f"{FIGURES}/jet_persistence/wrs_bars.png")

# props

In [ ]:
from scipy.stats import norm, chi2, t
from string import ascii_lowercase
data_vars = [
    "mean_lon",
    "mean_lat",
    "mean_theta",
    "mean_lev",
    "mean_s",
    "width",
    "tilt",
    "waviness1",
    "waviness2",
    "wavinessDC16",
    "wavinessFV15",
    "mean_lat_var",
    "mean_s_var",
    "is_polar",
    "int",
    "int_over_europe"
]
mode_dict = {
    "": "Connected components",
    "catd": "Full distance",
    "com": "COM distance",
}

def func(col):
    if ':' in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).var()
    return pl.col(col).mean()


def mean_confidence(col: pl.Series, q: float) -> pl.Series:
    n = (col.is_not_null() & col.is_not_nan()).sum()
    mu = col.mean()
    s_sq = col.var()
    if s_sq is None:
        return None
    s = np.sqrt(s_sq)
    sign = 1 - 2 * int(q < 0.5)
    q = q if q < 0.5 else 1 - q
    if n > 10:
        to_ret = mu + sign * np.abs(norm.ppf(q=q)) / np.sqrt(n) * s
    else:
        to_ret = mu + sign * s / np.sqrt(n) * t.ppf(q=1 - q, df=n - 1)
    to_ret = np.clip(to_ret, mu - 5 * s, mu + 5 * s)
    return to_ret


def var_confidence(col: pl.Series, q) -> float:
    n = (col.is_not_null() & col.is_not_nan()).sum()
    s_sq = col.var()
    if s_sq is None:
        return None
    sign = 1 - 2 * int(q < 0.5)
    if n > 50:
        q = q if q < 0.5 else 1 - q
        to_ret = s_sq + sign * np.sqrt(2 / n) * np.abs(norm.ppf(q)) * s_sq
    else:
        to_ret = (n - 1) * s_sq / chi2.ppf(1 - q, df=n - 1)
    return np.clip(to_ret, 0, s_sq * 2)
        
    
def func_q(col, q):
    if ':' in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).map_batches(partial(var_confidence, q=q), return_dtype=pl.Float64(), returns_scalar=True)
    return pl.col(col).map_batches(partial(mean_confidence, q=q), return_dtype=pl.Float64(), returns_scalar=True)
    

q_mean = 1e-15
for spell_of in ["STJ" ,"EDJ"]:
    spells_from_jet = spells_list[spell_of]
    n_spells = spells_from_jet["spell"].n_unique()
    props_masked = mask_from_spells_pl(spells_from_jet, phat_props_catd, time_before=datetime.timedelta(days=4))
    props_masked = props_masked.filter(pl.col("spell").n_unique().over("relative_index") > 14)
    aggs = {col: func(col) for col in data_vars}
    aggs = aggs | {f"{col}_10": func_q(col, q_mean) for col in data_vars}
    aggs = aggs | {f"{col}_90": func_q(col, 1 - q_mean) for col in data_vars}
    explode_list = [f"{col}_10" for col in data_vars] + [f"{col}_90" for col in data_vars]
    aggs = aggs | {"alive": pl.col("time").len()}
    mean_ps = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(**aggs)
    aggs_ = {col: func_q(col, 0.95) for col in data_vars}
    q25 = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(**aggs_)
    aggs_ = {col: func_q(col, 0.05) for col in data_vars}
    q75 = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(**aggs_)
    fig, axes = plt.subplots(4, 4, figsize=(11, 11), tight_layout=False, sharex="all")
    axes = axes.ravel()
    means = phat_props_catd_summer.group_by("jet", maintain_order=True).agg(**aggs)
    alive_spells = props_masked.group_by("relative_index").agg(pl.col("spell").n_unique()).sort("relative_index")["spell"].to_numpy()
    for j, jet in enumerate(["STJ", "EDJ"]):
        to_plot = mean_ps.filter(pl.col("jet") == jet)
        q25_ = q25.filter(pl.col("jet") == jet)
        q75_ = q75.filter(pl.col("jet") == jet)
        x = to_plot["relative_index"].unique().to_numpy() / 4
        for ax, data_var, letter in zip(axes, data_vars, ascii_lowercase):
            factor = 1e9 if data_var in ["int_over_europe", "int"] else 1
            factor = 1e5 if data_var == "width" else factor
            ax.plot(x, to_plot[data_var] / factor, color=COLORS[2 - j], lw=2.5)
            ax.fill_between(x, q25_[data_var] / factor, q75_[data_var] / factor, color=COLORS[2 - j], alpha=0.4)
            mean = means.filter(pl.col("jet")==jet)[data_var].item() / factor
            q10 = means.filter(pl.col("jet")==jet)[f"{data_var}_10"].item() / factor
            q90 = means.filter(pl.col("jet")==jet)[f"{data_var}_90"].item() / factor
            ax.plot([x[0], x[-1]], [mean, mean], color=COLORS[2 - j], ls="dashed", lw=2)
            if j == 0:
                factor_str = "" if factor == 1 else rf"$10^{int(np.log10(factor))} \times $"
                ax.set_title(
                    rf"{letter}) {PRETTIER_VARNAME.get(data_var, data_var)} [{factor_str}{UNITS.get(data_var, '~')}]"
                )
            ax.yaxis.set_major_locator(MaxNLocator(4, integer=True))
    for i, ax in enumerate(axes):
        ax.axvline(0, zorder=1, color="black", lw=2)
        if i > 11:
            ax.set_xlabel("Relative time around onset [days]", color="black")
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        ybounds = [ylim[0] - 0.05 * (ylim[1] - ylim[0]), ylim[0] + 0.05 * (ylim[1] - ylim[0])]
        im = ax.pcolormesh(
            x, ybounds, alive_spells[None, :-1], zorder=-10,
            cmap=colormaps.greys, alpha=0.7, vmin=0
        )
    fig.set_constrained_layout(True)
    mode = mode_dict[spell_of[4:]]
    fig.suptitle(f"Persistent episodes of the {spell_of[:3]}. Mode: {mode}. {props_masked['spell'].n_unique()} spells")
    fig.savefig(f"{FIGURES}/jet_persistence/{spell_of}_props_{n_spells}spells.png")
    # plt.close()

In [ ]:
from scipy.stats import norm, chi2, t
from string import ascii_lowercase
data_vars = [
    "mean_lat",
    "mean_theta",
    "mean_s",
    # "width",
    "tilt",
    "wavinessR16",
    # "mean_lat_var",
    "mean_s_var",
    # "is_polar",
]
mode_dict = {
    "": "Connected components",
    "catd": "Full distance",
    "com": "COM distance",
}

def func(col):
    if ':' in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).var()
    return pl.col(col).mean()


def mean_confidence(col: pl.Series, q: float) -> pl.Series:
    n = (col.is_not_null() & col.is_not_nan()).sum()
    mu = col.mean()
    s_sq = col.var()
    if s_sq is None:
        return None
    s = np.sqrt(s_sq)
    sign = 1 - 2 * int(q < 0.5)
    q = q if q < 0.5 else 1 - q
    if n > 10:
        to_ret = mu + sign * np.abs(norm.ppf(q=q)) / np.sqrt(n) * s
    else:
        to_ret = mu + sign * s / np.sqrt(n) * t.ppf(q=1 - q, df=n - 1)
    to_ret = np.clip(to_ret, mu - 5 * s, mu + 5 * s)
    return to_ret


def var_confidence(col: pl.Series, q) -> float:
    n = (col.is_not_null() & col.is_not_nan()).sum()
    s_sq = col.var()
    if s_sq is None:
        return None
    sign = 1 - 2 * int(q < 0.5)
    if n > 50:
        q = q if q < 0.5 else 1 - q
        to_ret = s_sq + sign * np.sqrt(2 / n) * np.abs(norm.ppf(q)) * s_sq
    else:
        to_ret = (n - 1) * s_sq / chi2.ppf(1 - q, df=n - 1)
    return np.clip(to_ret, 0, s_sq * 2)
        
    
def func_q(col, q):
    if ':' in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).map_batches(partial(var_confidence, q=q), return_dtype=pl.Float64(), returns_scalar=True)
    return pl.col(col).map_batches(partial(mean_confidence, q=q), return_dtype=pl.Float64(), returns_scalar=True)
    

q_mean = 1e-15
for group in [["STJ" ,"EDJ"]]:
    group_id = mode_dict[group[0][4:]]
    bigfig = plt.figure(figsize=(14, 11), constrained_layout=True)
    subfigs = bigfig.subfigures(1, 2)
    for spell_of, fig in zip(group, subfigs):
        spells_from_jet = spells_list[spell_of]
        props_masked = mask_from_spells_pl(spells_from_jet, phat_props_catd, time_before=datetime.timedelta(days=4))
        props_masked = props_masked.filter(pl.col("spell").n_unique().over("relative_index") > 15)
        aggs = {col: func(col) for col in data_vars}
        aggs = aggs | {f"{col}_10": func_q(col, q_mean) for col in data_vars}
        aggs = aggs | {f"{col}_90": func_q(col, 1 - q_mean) for col in data_vars}
        explode_list = [f"{col}_10" for col in data_vars] + [f"{col}_90" for col in data_vars]
        aggs = aggs | {"alive": pl.col("time").len()}
        mean_ps = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(**aggs)
        aggs_ = {col: func_q(col, 0.95) for col in data_vars}
        q25 = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(**aggs_)
        aggs_ = {col: func_q(col, 0.05) for col in data_vars}
        q75 = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(**aggs_)
        # fig, axes = plt.subplots(3, 3, figsize=(11, 11), tight_layout=False, sharex="all")
        axes = fig.subplots(3, 2, sharex="all")
        axes = axes.ravel()
        means = phat_props_catd_summer.group_by("jet", maintain_order=True).agg(**aggs)
        alive_spells = props_masked.group_by("relative_index").agg(pl.col("spell").n_unique()).sort("relative_index")["spell"].to_numpy()
        ascii_lowercase_ = ascii_lowercase[int(spell_of[:3] == "EDJ") * len(axes):]
        for j, jet in enumerate(["STJ", "EDJ"]):
            to_plot = mean_ps.filter(pl.col("jet") == jet)
            q25_ = q25.filter(pl.col("jet") == jet)
            q75_ = q75.filter(pl.col("jet") == jet)
            x = to_plot["relative_index"].unique().to_numpy() / 4
            for ax, data_var, letter in zip(axes, data_vars, ascii_lowercase_):
                factor = 1e9 if data_var in ["int_over_europe", "int"] else 1
                factor = 1e5 if data_var == "width" else factor
                ax.plot(x, to_plot[data_var] / factor, color=COLORS[2 - j], lw=2.5)
                ax.fill_between(x, q25_[data_var] / factor, q75_[data_var] / factor, color=COLORS[2 - j], alpha=0.4)
                mean = means.filter(pl.col("jet")==jet)[data_var].item() / factor
                q10 = means.filter(pl.col("jet")==jet)[f"{data_var}_10"].item() / factor
                q90 = means.filter(pl.col("jet")==jet)[f"{data_var}_90"].item() / factor
                ax.plot([x[0], x[-1]], [mean, mean], color=COLORS[2 - j], ls="dashed", lw=2)
                if j == 0:
                    factor_str = "" if factor == 1 else rf"$10^{int(np.log10(factor))} \times $"
                    ax.set_title(
                        rf"{letter}) {PRETTIER_VARNAME.get(data_var, data_var)} [{factor_str}{UNITS.get(data_var, '~')}]"
                    )
                ax.yaxis.set_major_locator(MaxNLocator(4, integer=True))
        for i, ax in enumerate(axes):
            ax.axvline(0, zorder=1, color="black", lw=2)
            if i > 3:
                ax.set_xlabel("Relative time around onset [days]", color="black")
            xlim = ax.get_xlim()
            ylim = ax.get_ylim()
            ybounds = [ylim[0] - 0.05 * (ylim[1] - ylim[0]), ylim[0] + 0.05 * (ylim[1] - ylim[0])]
            im = ax.pcolormesh(
                x, ybounds, alive_spells[None, :-1], zorder=-10,
                cmap=colormaps.greys, alpha=0.7
            )
        mode = mode_dict[spell_of[4:]]
        fig.suptitle(f"Persistent episodes of the {spell_of[:3]}. {props_masked['spell'].n_unique()} spells")
    bigfig.savefig(f"{FIGURES}/jet_persistence/both_props_30spells.png")
    # plt.close()

# realspace from data

In [ ]:
figure_data = f"{FIGURES}/jet_persistence/figure_data/realspace_fig6"
data_contour = xr.open_dataarray(f"{figure_data}/contour.nc")
data_signif = xr.open_dataset(f"{figure_data}/signifs.nc")
data_wind = xr.open_dataset(f"{figure_data}/wind.nc")
data_contourf = xr.open_dataset(f"{figure_data}/contourf.nc")

long_names = {
    "t2m": "2m temperature [K]",
    "t_up": "Upper level temperature [K]",
    "tp": "Daily accum. precip. [mm]",
    "apvs": r"Anticyclonic PV streamer freq [$\%$]",
    "cpvs": r"Cyclonic PV streamer freq [$\%$]",
    "theta": "Upper-level Potential temperature [K]",
    "s": r"Wind speed [$\mathrm{m.s}^{-1}$]",
    "theta": "Pot temp at 2 PVU [K]",
    "z200": "Z200 anomaly [m]",
    "z500": "Z500 anomaly [m]",
    "pv330": "PV at 330K [PVU]",
    "pv350": "PV at 350K [PVU]",
}

In [ ]:
from string import ascii_lowercase
from cartopy.mpl.geoaxes import GeoAxes
plt.rc("axes", titlesize=14)
cbar_kwargs = {"shrink": 1.0, "fraction": 0.11, "pad": 0.03}
plot_kwargs_1 = {
    "cmap": colormaps.BlWhRe, 
    "levels": [-1.2, -0.8, -.4, .4, 0.8, 1.2], 
    "transparify": 1, 
    "cbar_kwargs": cbar_kwargs
}
plot_kwargs_2 = {
    "cmap": colormaps.brbg,
    "levels": np.linspace(-2, 2, 9).tolist(),
    "transparify": 0, 
    "cbar_kwargs": cbar_kwargs,
}
stippling_kwargs = {
    "FDR": True,
    "invert": False,
    "linewidth": 0.7,
    "color": "black",
    "hatch": "xxx",
}
nrow, ncol = 1, 2
days_around = 3
n = nrow * ncol
cmap = colormaps.pastel
norm = BoundaryNorm(np.arange(cmap.N) + 0.5, cmap.N)
bigfig = plt.figure(figsize=(6.8, 6), constrained_layout=True)
subfigs = bigfig.subfigures(2, 1)
for varname, plot_kwargs, fig in zip(["t2m", "tp"], [plot_kwargs_1, plot_kwargs_2], subfigs):
    clu = Clusterplot(nrow, ncol, get_region(data_contourf[varname]), fig=fig)
    titles = []
    for j, jet in enumerate(spells_list):
        letter = ascii_lowercase[j % len(ascii_lowercase) + len(clu.axes) * int(varname == "tp")]
        n_spells = daily_spells_list[jet]["spell"].n_unique()
        titles.append(f"{letter}) {jet[:3]}, {n_spells} episodes")
    clu.add_contourf(data_contourf[varname], titles=titles, **plot_kwargs)
    clu.add_stippling(data_signif[varname].astype(bool), **stippling_kwargs)
    clu.add_contour(data_contour, levels=[-60, -20, 20, 60], linewidths=2., clabels=False)
    fig.suptitle(long_names[varname])
    
jets_on_mean = find_all_jets(data_wind)
for fig in subfigs:
    for ax, (_, jets) in zip(fig.axes, jets_on_mean.group_by("relative_index")):
        if not isinstance(ax, GeoAxes):
            continue
        for _, jet_ in jets.group_by("jet ID"):
            lo, la = jet_[["lon", "lat"]]
            if la.mean() > 40: 
                color = COLORS[1]
            else:
                color = COLORS[2]
            ax.plot(lo - clu.central_longitude, la, lw=2.5, color=color)
bigfig.savefig(f"{FIGURES}/jet_persistence/t2m_and_tp_realspace_both_30spells.png")

# temps ts

In [ ]:
fig, ax = plt.subplots()
year = 1982
jet = "EDJ"
jet_id = int(jet == "EDJ")
q9 = pers.filter(pl.col("jet ID") == jet_id)["pers"].quantile(0.5)
pers_ = pers.filter(pl.col("jet ID") == jet_id, pl.col("time").dt.year() == year, pl.col("time").dt.month() == 6)
filter_ = (
    (pers_["pers"] > q9)
    .rle()
    .to_frame()
    .unnest("pers")
    .with_columns(
        start=pl.lit(0).append(
            pl.col("len").cum_sum().slice(0, pl.col("len").len() - 1)
        )
    )
    .with_columns(index=pl.int_ranges(pl.col("start"), pl.col("start") + pl.col("len")))
    .explode("index")
    .with_columns(condition=pl.col("value") & (pl.col("len") >= 24))
)

ax.plot(pers_["time"][[0, -1]], [q9, q9], color=COLORS[1], ls="dashed")
ax.plot(pers_["time"], pers_["pers"], color=COLORS[1], lw=2)
ax.fill_between(
    pers_["time"], q9, pers_["pers"], where=filter_["condition"], color=COLORS[1], alpha=.7
)
ax.xaxis.set_tick_params(labelrotation=20)
ax.set_ylabel("Persistence [s/m]")
# fig.savefig(f"{FIGURES}/jet_persistence/new_demo.png")

In [ ]:
filter_.with_columns(condition=pl.col("value") & (pl.col("len") >= 12))["condition"].any()

In [ ]:
filter_ = (pers_["pers"]>q9).rle().to_frame().unnest("pers").with_columns(start=pl.lit(0).append(pl.col("len").cum_sum().slice(0, pl.col("len").len() - 1))).with_columns(index=pl.int_ranges(pl.col("start"), pl.col("start") + pl.col("len"))).explode("index")

In [ ]:
from matplotlib.dates import DateFormatter
from matplotlib.style import context as mstylecont, available as mstyleavail

year = 2018
region_T_anom = pl.read_parquet(basepath.joinpath("regionalized_T_anom.parquet"))
cross_upd, sommary_comp = connected_from_cross(phat_jets, cross_phat, dis_polar_thresh=0.15, dist_thresh=2e5)
this_summer_filter = pl.col("time").dt.year() == year, pl.col("time").is_in(summer.implode())
those_comps = sommary_comp.filter(this_summer_filter)
pers = pers_from_cross_catd(cross_catd, summer)
pers_this_period = pers.filter(this_summer_filter)
props_this_period = phat_props_catd_summer.filter(this_summer_filter)
pers_period = region_T_anom.filter(pl.col("time").dt.year() == year)

spell_stj = spells_list['STJ_catd'].filter(pl.col("time").dt.year() == year)
spell_edj = spells_list['EDJ_catd'].filter(pl.col("time").dt.year() == year)

fig, (ax, ax0, ax00, ax1, ax2, ax3) = plt.subplots(6, 1, gridspec_kw=dict(hspace=0), height_ratios=[0.7, 0.1, 0.1, 0.2, 0.2, 0.2], sharex="all", figsize=(12, 4))
colors = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]

for reg, subdf in pers_period.sort("region").group_by("region", maintain_order=True):
    reg = reg[0]
    ax.plot(subdf["time"], subdf["t2m"], color=colors[reg - 1], lw=2.5, label=DUNCANS_REGIONS_NAMES[reg - 1])
ax.xaxis.set_major_formatter(DateFormatter(r"%d/%m"))
ylim = ax.get_ylim()
ylim1 = ax0.get_ylim()
for spell_id, subspelldf in spell_stj.group_by("spell"):
    ax0.fill(subspelldf["time"][0, -1, -1, 0, 0].to_numpy(), np.array(ylim1)[[0, 0, 1, 1, 0]], color=COLORS[2])
    ax0.text(subspelldf["time"].mean(), 0.4, "STJ", ha="center", va="center", fontsize=14)
    ax.vlines(subspelldf["time"][0, -1].to_numpy(), *ylim, color="grey", ls="dashed")
    ax0.vlines(subspelldf["time"][0, -1].to_numpy(), *ylim1, color="grey", ls="dashed")
for spell_id, subspelldf in spell_edj.group_by("spell"):
    ax00.fill(subspelldf["time"][0, -1, -1, 0, 0].to_numpy(), np.array(ylim1)[[0, 0, 1, 1, 0]], color=COLORS[1])
    ax00.text(subspelldf["time"].mean(), 0.4, "EDJ", ha="center", va="center", fontsize=14)
    ax.vlines(subspelldf["time"][0, -1].to_numpy(), *ylim, color="grey", ls="dashed")
    ax0.vlines(subspelldf["time"][0, -1].to_numpy(), *ylim1, color="grey", ls="dashed", zorder=12)
    ax00.vlines(subspelldf["time"][0, -1].to_numpy(), *ylim1, color="grey", ls="dashed")
ax0.set_yticks([])
ax00.set_yticks([])
ax.set_ylim(ylim[0], ylim[1] + 4)
ax.legend(ncol=6, loc="upper center")

for j, (jet, q_pers, q_com) in enumerate(zip(["STJ", "EDJ"], [0.99, 0.95], [0.4, 0.4])):
    pers_ = pers_this_period.filter(pl.col("spell_of") == jet)
    ax1.plot(pers_["time"], pers_["pers"], color=COLORS[2 - j])
    summer_pers_ = pers.filter(pl.col("spell_of") == jet, pl.col("time").is_in(summer.implode()))["pers"].quantile(q_pers)
    props_ = props_this_period.filter(pl.col("jet") == jet)
    ax1.plot(props_["time"][0, -1].to_numpy(), [summer_pers_, summer_pers_], color=COLORS[2 - j], ls="dashed")
    ax2.plot(props_["time"], 1 / props_["com_speed"], color=COLORS[2 - j])
    com_speed_mean = phat_props_catd_summer.filter(pl.col("jet") == jet)["com_speed"].replace(np.inf, 1000).quantile(q_com)
    ax2.plot(props_["time"][0, -1], [1 / com_speed_mean, 1 / com_speed_mean], color=COLORS[2 - j], ls="--")


for comp_number, comp in those_comps.group_by("spell"):
    comp = comp.with_columns(pers=persistence_expr())
    j = comp["is_polar"].mean() > 0.5
    ax3.plot(comp["time"], comp["pers"], color=COLORS[2 - j], lw=2)
# fig.savefig(f"{FIGURES}/jet_persistence/case_study.png")

In [ ]:
from matplotlib.dates import DateFormatter
from matplotlib.style import context as mstylecont, available as mstyleavail

year = 1976
region_T_anom = pl.read_parquet(basepath.joinpath("regionalized_T_anom.parquet"))
cross_upd, sommary_comp = connected_from_cross(phat_jets, cross_phat, dis_polar_thresh=0.15, dist_thresh=2e5)
this_summer_filter = pl.col("time").dt.year() == year, pl.col("time").is_in(summer.implode())
those_comps = sommary_comp.filter(this_summer_filter)
pers = pers_from_cross_catd(cross_catd, summer)
pers_this_period = pers.filter(this_summer_filter)
this_summer = summer.filter(summer.dt.year() == year).rename("time").to_frame()
props_this_period = phat_props_catd_summer.filter(this_summer_filter)
pers_period = region_T_anom.filter(pl.col("time").dt.year() == year)

spell_stj = spells_list['STJ_catd'].filter(pl.col("time").dt.year() == year)
spell_edj = spells_list['EDJ_catd'].filter(pl.col("time").dt.year() == year)

fig, (ax, ax0, ax00, ax1) = plt.subplots(4, 1, gridspec_kw=dict(hspace=0), height_ratios=[0.7, 0.1, 0.1, 0.2], sharex="all", figsize=(12, 4))
colors = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]

for reg, subdf in pers_period.sort("region").group_by("region", maintain_order=True):
    reg = reg[0]
    ax.plot(subdf["time"], subdf["t2m"], color=colors[reg - 1], lw=2.5, label=DUNCANS_REGIONS_NAMES[reg - 1])
ax.xaxis.set_major_formatter(DateFormatter(r"%d/%m"))
ylim = ax.get_ylim()
ylim1 = ax0.get_ylim()
for spell_id, subspelldf in spell_stj.group_by("spell"):
    ax0.fill(subspelldf["time"][0, -1, -1, 0, 0].to_numpy(), np.array(ylim1)[[0, 0, 1, 1, 0]], color=COLORS[2])
    ax0.text(subspelldf["time"].mean(), 0.4, "STJ", ha="center", va="center", fontsize=14)
    ax.vlines(subspelldf["time"][0, -1].to_numpy(), *ylim, color="grey", ls="dashed")
    ax0.vlines(subspelldf["time"][0, -1].to_numpy(), *ylim1, color="grey", ls="dashed")
for spell_id, subspelldf in spell_edj.group_by("spell"):
    ax00.fill(subspelldf["time"][0, -1, -1, 0, 0].to_numpy(), np.array(ylim1)[[0, 0, 1, 1, 0]], color=COLORS[1])
    ax00.text(subspelldf["time"].mean(), 0.4, "EDJ", ha="center", va="center", fontsize=14)
    ax.vlines(subspelldf["time"][0, -1].to_numpy(), *ylim, color="grey", ls="dashed")
    ax0.vlines(subspelldf["time"][0, -1].to_numpy(), *ylim1, color="grey", ls="dashed", zorder=12)
    ax00.vlines(subspelldf["time"][0, -1].to_numpy(), *ylim1, color="grey", ls="dashed")
ax0.set_yticks([])
ax00.set_yticks([])
ax.set_ylim(ylim[0], ylim[1] + 4)
ax.legend(ncol=6, loc="upper center")

for j, (jet, q_pers, q_com) in enumerate(zip(["STJ", "EDJ"], [0.8, 0.7], [0.4, 0.4])):
    pers_ = pers_this_period.filter(pl.col("spell_of") == jet)
    pers_ = this_summer.join(pers_, on="time", how="left")
    props_ = props_this_period.filter(pl.col("jet") == jet)
    props_ = this_summer.join(props_, on="time", how="left")

    pers_.with_columns(pers=pl.when(props_["mean_lat"].is_null()).then(pl.lit(None)).otherwise("pers"))
    ax1.plot(pers_["time"], pers_["pers"], color=COLORS[2 - j])
    summer_pers_ = pers.filter(pl.col("spell_of") == jet, pl.col("time").is_in(summer.implode()))["pers"].quantile(q_pers)
    ax1.plot(pers_["time"][0, -1].to_numpy(), [summer_pers_, summer_pers_], color=COLORS[2 - j], ls="dashed")
fig.savefig(f"{FIGURES}/jet_persistence/case_study{year}.png")

## cat

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, rgb_to_hsv, hsv_to_rgb
from tqdm import trange
from jetutils.jet_finding import extract_features
from jetutils.definitions import MONTH_NAMES

xys = []
all_jets = all_jets_one_df

xys = np.array(xys)
fig, axes = plt.subplots(3, 4, figsize=(11, 9), tight_layout=True, sharex="all", sharey="all")
axes = axes.ravel()
pair = ["s", "theta", "is_polar"]
cmap = LinearSegmentedColormap.from_list("pinkredpurple", [COLORS[2], COLORS_EXT[10], COLORS[1]])
bins = np.linspace(0, 1, 31)
for month in trange(1, 13):
    ax = axes[month - 1]
    X = extract_features(all_jets, pair, season=month)
    probas = X[pair[2]]
    center_stj = X.filter(pl.col("is_polar") < 0.3).mean()
    center_edj = X.filter(pl.col("is_polar") > 0.7).mean()
    X1D = X["is_polar"]
    
    im1 = ax.hexbin(X[pair[0]], X[pair[1]], cmap=colormaps.gray_r, gridsize=25)
    im2 = ax.hexbin(X[pair[0]], X[pair[1]], C=probas, cmap=colormaps.gray_r, gridsize=25)
    
    plt.draw()
        
    offsets1 = np.asarray(list(map(tuple, im1.get_offsets())), dtype="f, f")
    offsets2 = np.asarray(list(map(tuple, im2.get_offsets())), dtype="f, f")
    mask12 = np.isin(offsets1, offsets2)
    colors = cmap(im2.get_array())
    colors = rgb_to_hsv(colors[:, :3])
    min_s, max_s = 0., 1.0
    min_v, max_v = 0.75, 1.
    scaling = np.clip(im1.get_array()[mask12] / im1.get_array()[mask12].max() * 1.1, 0, 1)
    f = lambda x: np.sqrt(x)
    colors[:, 1] = min_s + scaling * (max_s - min_s)
    colors[:, 2] = max_v - scaling * (max_v - min_v)
    colors = hsv_to_rgb(colors)
    im2.set_array(None)
    im2.set_facecolor(colors)
    # im2.set_linewidths(0.2)
    im2.set_linewidths(4 * (1 - f(scaling)) ** 2)
    im2.set_edgecolor(colormaps.greys(scaling))
    im2 = ax.add_collection(im2)
        
    ax.set_xlabel(pair[0])
    ax.set_ylabel(pair[1])
    if pair[0] in ["lev", "theta"]:
        ax.invert_xaxis()
    elif pair[1] in ["lev", "theta"]:
        ax.invert_yaxis() 
        
    ax.set_title(MONTH_NAMES[month - 1])
    ax.scatter(*pl.concat([center_stj, center_edj])[pair[:2]].to_numpy().T, facecolor="black", edgecolor="white", marker="X", linewidths=1, s=45)
    iax = ax.inset_axes([0.65, 0.2, 0.4, 0.5])
    X1D = np.clip(X1D, 0, 1)
    iax.hist(X1D, bins=bins, alpha=0.5, color="black")
    iax.hist(X1D[probas > 0.5], bins=bins, alpha=0.5, color=COLORS[1])
    iax.hist(X1D[probas < 0.5], bins=bins, alpha=0.5, color=COLORS[2])
    iax.set_xticks([0, 0.5, 1])
    iax.set_yticks([])
    iax.spines[["left", "right", "top"]].set_visible(False)
    plt.draw()
# fig.savefig(f"{FIGURES}/gmix_demo.png")

# rwb

In [ ]:
basepath_rwb = Path("/Users/bandelol/Documents/code_local/data/rwb_index")
for f in basepath_rwb.glob("era5_pv_streamers_???K_1959-2022.parquet"):
    isentrope = int(f.name.split("_")[3][:3])
    df = pl.read_parquet(f)
    df = df.filter(
        pl.col("com").list.get(0) > -100,
        pl.col("com").list.get(0) < 60,
        pl.col("com").list.get(1) > 0,
    )
    if isentrope == 340:
        break

In [ ]:
basepath_rwb = Path("/Users/bandelol/Documents/code_local/data/rwb_index")
# for f in basepath_rwb.glob("era5_pv_streamers_???K_1959-2022.parquet"):
for isentrope in [320, 330, 340, 350]:
    f = basepath_rwb.joinpath(f"era5_pv_streamers_{isentrope}K_1959-2022.parquet")
    # isentrope = int(f.name.split("_")[3][:3])
    df = pl.read_parquet(f)
    df = df.filter(
        pl.col("com").list.get(0) > -100,
        pl.col("com").list.get(0) < 60,
        pl.col("com").list.get(1) > 0,
    )
    for thresh in [0.]:
        large = pl.col("event_area") > pl.col("event_area").quantile(thresh)
        strato = df.filter(pl.col("mean_var") > 2)
        anticyclonic = strato.filter(pl.col("intensity") > 0).filter(large)
        cyclonic = strato.filter(pl.col("intensity") < 0).filter(large)

        anti_reduced = anticyclonic[["date", "level", "com"]].rename({"date": "time", "level": "event_anti"}).with_columns(pl.col("time").cast(pl.Datetime("ms")), pl.col("event_anti").cast(pl.Boolean())).drop("com")
        cycl_reduced = cyclonic[["date", "level", "com"]].rename({"date": "time", "level": "event_cycl"}).with_columns(pl.col("time").cast(pl.Datetime("ms")), pl.col("event_cycl").cast(pl.Boolean())).drop("com")

        base_anti, base_cycl = summer_filter.join(anti_reduced, on="time", how="left").join(cycl_reduced, on="time", how="left").fill_null(False).select(pl.col("event_anti").mean(), pl.col("event_cycl").mean()).to_numpy()[0]
        fig, axes = plt.subplots(2, 1, sharex=True)
        fig.suptitle(f"{isentrope}, {thresh}")
        for spell_of, ax in zip(["STJ", "EDJ"], axes):
            spells = spells_list[f"{spell_of}"]
            n_bootstraps = 100
            bs_ts = create_bootstrapped_times(spells, summer, n_bootstraps=n_bootstraps)
            bs_ts = bs_ts.join(anti_reduced, on="time", how="left").join(cycl_reduced, on="time", how="left").fill_null(False)
            bs_ts = bs_ts.group_by("relative_index", "sample_index").agg(pl.col("event_anti").mean(), pl.col("event_cycl").mean())
            bs_ts = bs_ts.group_by("relative_index").agg(
                *[pl.col(f"event_{type_}").filter(pl.col("sample_index") == n_bootstraps).first() for type_ in ["anti", "cycl"]],
                *[((pl.col(f"event_{type_}").rank() - 1) / n_bootstraps).filter(pl.col("sample_index") == n_bootstraps).alias(f"pval_{type_}").first() for type_ in ["anti", "cycl"]],
            ).sort("relative_index")

            ax.plot(bs_ts["relative_index"], bs_ts["event_anti"], lw=2, color=colormaps.bold(0))
            filtered = bs_ts.filter((pl.col("pval_anti") > 0.975) | (pl.col("pval_anti") < 0.025))
            ax.scatter(filtered["relative_index"], filtered["event_anti"], color=colormaps.bold(0))
            ax.plot(bs_ts["relative_index"][[0, -1]], [base_anti, base_anti], color=colormaps.bold(0), lw=2, ls="dashed")
            ax.plot(bs_ts["relative_index"], bs_ts["event_cycl"], lw=2, color=colormaps.bold(0.3))
            filtered = bs_ts.filter((pl.col("pval_cycl") > 0.975) | (pl.col("pval_cycl") < 0.025))
            ax.scatter(filtered["relative_index"], filtered["event_cycl"], color=colormaps.bold(0.3))
            ax.plot(bs_ts["relative_index"][[0, -1]], [base_cycl, base_cycl], color=colormaps.bold(0.3), lw=2, ls="dashed")
            ax.set_ylim([-0.04, 1.04])
            ax.set_title(spell_of)

In [ ]:
for df_reduced, type_ in zip(
    [anti_reduced, cycl_reduced],
    ["anti", "cycl"]
):
    df_reduced_ = ALL_TIMES.join(df_reduced, on="time", how="left").fill_null(False)
    huh = (
        df_reduced_
        .select(pl.col(f"event_{type_}").rle())
        .unnest(f"event_{type_}")
        .with_columns(
            start=pl.lit(0).append(pl.col("len").cum_sum().slice(0, pl.col("len").len() - 1))
        )
    )
    huh = (
        ALL_TIMES
        .with_row_index()
        .join(explode_rle(huh), on="index")
    )
    do_quantiles = np.linspace(0, 1, 101).tolist()
    quantiles = summer_filter.join(huh, on="time").filter(~pl.col("value")).select(**{f"len_{q=}": pl.col("len").quantile(q) for q in do_quantiles})
    for jet in ["STJ", "EDJ"]:
        spells = spells_list[f"{jet}_catd"]
        (
            spells
            .filter(pl.col("relative_index") < 12)
            .join(huh, on="time")
            .filter(~pl.col("value"))
            .select(
                pl.col("len_right").mean(), 
                pl.lit(pl.Series("hoho", quantiles)).search_sorted(pl.col("len_right"))
            )
        )

In [ ]:
(
    spells
    .filter(pl.col("relative_index") < 12)
    .join(huh, on="time")
    .filter(~pl.col("value"))
)

In [ ]:
quantiles.transpose()["column_0"]

In [ ]:
pl.Series("hoho", quantiles.to_numpy()).explode()

In [ ]:
(
    spells
    .filter(pl.col("relative_index") < 12)
    .join(huh, on="time")
    .filter(~pl.col("value"))
    .select(
        # pl.col("len_right").mean(), 
        pl.lit(quantiles.transpose()["column_0"]).search_sorted(pl.col("len_right")) / 100
    )
)["column_0"].unique().to_list()

In [ ]:
spells_list["EDJ_catd"].filter(pl.col("relative_index") < 12).join(huh, on="time").filter(~pl.col("value"))["len_right"].select(pl.col("len_right").mean(), **{f"len_{q=}": pl.col("len_right").quantile(q) for q in [0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99]})